In [12]:
# %pip install "arize-phoenix>=10.0.0" "arize-phoenix-evals>=0.20.6" 'httpx<0.28' duckdb datasets pyarrow "pydantic>=2.0.0"  openinference-instrumentation openinference-instrumentation-langchain nest_asyncio langchain-google-vertexai google-cloud-aiplatform python-dotenv duckdb --quiet


In [13]:
# %pip install langgraph

In [14]:
# %pip install --upgrade "arize-phoenix>=10.0.0"
# %pip install --upgrade "arize-phoenix-evals>=0.20.6"


In [15]:
from google.cloud.aiplatform import init as init_vertexai
PROJECT_ID = "seequent-labs-dev"  
LOCATION = "us-central1"
init_vertexai(project=PROJECT_ID, location=LOCATION)

In [16]:
from langchain_google_vertexai import ChatVertexAI
model_name = "gemini-2.5-flash"

model = ChatVertexAI(
    model=model_name,
    temperature=0,
)
client = model


In [17]:
from phoenix.evals import GeminiModel

# Initialize the Gemini client for Phoenix Evals
eval_model = GeminiModel(
    model="gemini-1.5-pro",  # or "gemini-2.0-flash", etc.
    project=PROJECT_ID,
    location=LOCATION
)


/Users/stepan.lavrinenko/phoenix/.venv/lib/python3.12/site-packages/vertexai/generative_models/_generative_models.py:433: UserWarning: This feature is deprecated as of June 24, 2025 and will be removed on June 24, 2026. For details, see https://cloud.google.com/vertex-ai/generative-ai/docs/deprecations/genai-vertexai-sdk.
  warning_logs.show_deprecation_warning()


<center>
    <p style="text-align:center">
        <img alt="phoenix logo" src="https://storage.googleapis.com/arize-assets/phoenix/assets/phoenix-logo-light.svg" width="200"/>
        <br>
        <a href="https://arize.com/docs/phoenix/">Docs</a>
        |
        <a href="https://github.com/Arize-ai/phoenix">GitHub</a>
        |
        <a href="https://arize-ai.slack.com/join/shared_invite/zt-2w57bhem8-hq24MB6u7yE_ZF_ilOYSBw#/shared-invite/email">Community</a>
    </p>
</center>
<h1 align="center">Evaluating an Agent</h1>

This notebook serves as an end-to-end example of how to trace and evaluate an agent. The example uses a "talk-to-your-data" agent as its example.

The notebook includes:
* Manually instrumenting an agent using Phoenix decorators
* Evaluating function calling accuracy using LLM as a Judge
* Evaluating function calling accuracy by comparing to ground truth
* Evaluating SQL query generation
* Evaluating Python code generation
* Evaluating the path of an agent

## Install Dependencies, Import Libraries, Set API Keys

In [18]:
# from phoenix.evals.models import VertexAIModel
# from phoenix.evals.templates import PromptTemplate

# # Use Vertex AI model for evaluations

# model=VertexAIModel(
#     model="gemini-2.0-flash",
#     project=PROJECT_ID,
#     location=LOCATION
# ),


In [19]:
project_name = "talk-to-your-data-agent"


In [20]:
import os

os.environ["PHOENIX_COLLECTOR_ENDPOINT"] = "http://localhost:6006"


from phoenix.otel import register
os.environ["PHOENIX_PROJECT_NAME"] = project_name

# configure the Phoenix tracer
tracer_provider = register(
  project_name=project_name, # Default is 'default'
  auto_instrument=True # Auto-instrument your app based on installed OI dependencies
)

  # Import the automatic instrumentor from OpenInference
from openinference.instrumentation.langchain import LangChainInstrumentor

# Finish automatic instrumentation
LangChainInstrumentor().instrument(tracer_provider=tracer_provider)
from openinference.instrumentation import using_attributes, using_tags


tracer = tracer_provider.get_tracer(__name__)


Overriding of current TracerProvider is not allowed
DependencyConflict: requested: "openai >= 1.69.0" but found: "None"
Attempting to instrument while already instrumented
Attempting to instrument while already instrumented


🔭 OpenTelemetry Tracing Details 🔭
|  Phoenix Project: talk-to-your-data-agent
|  Span Processor: SimpleSpanProcessor
|  Collector Endpoint: localhost:4317
|  Transport: gRPC
|  Transport Headers: {'user-agent': '****'}
|  
|  Using a default SpanProcessor. `add_span_processor` will overwrite this default.
|  
|  ⚠️ WARNING: It is strongly advised to use a BatchSpanProcessor in production environments.
|  
|  `register` has set this TracerProvider as the global OpenTelemetry default.
|  To disable this behavior, call `register` with `set_global_tracer_provider=False`.



In [22]:
import os
from typing import List, Dict, Any, Optional
from langchain_google_vertexai import ChatVertexAI
from langchain_core.messages import HumanMessage, AIMessage, SystemMessage
from langchain_core.tools import tool
from langchain_core.runnables import RunnablePassthrough
from langgraph.graph import StateGraph, END, START  # Removed incorrect Graph import
from langgraph.prebuilt import ToolNode
from phoenix.evals import GeminiModel
from phoenix.otel import register
from openinference.instrumentation.langchain import LangChainInstrumentor
from openinference.instrumentation import using_attributes, using_tags
import json

# Phoenix configuration

project_name = "evaluate_agent_project"

os.environ["PHOENIX_COLLECTOR_ENDPOINT"] = "http://localhost:6006"
os.environ["PHOENIX_PROJECT_NAME"] = project_name

# Configure the Phoenix tracer
tracer_provider = register(
    project_name=project_name,  # Default is 'default'
    auto_instrument=True  # Auto-instrument your app based on installed OI dependencies
)

# Import the automatic instrumentor from OpenInference
LangChainInstrumentor().instrument(tracer_provider=tracer_provider)

tracer = tracer_provider.get_tracer(__name__)

# # Initialize the Gemini client for Phoenix Evals
# eval_model = GeminiModel(
#     model="gemini-1.5-pro",  # or "gemini-2.0-flash", etc.
#     project=PROJECT_ID,
#     location=LOCATION
# )

# Initialize ChatVertexAI client
client = eval_model = ChatVertexAI(
    model_name="gemini-1.5-pro",
    project=PROJECT_ID,
    location=LOCATION,
    temperature=0.1
)

# Define evaluation tools
@tool
def evaluate_response_quality(response: str, expected_criteria: str) -> Dict[str, Any]:
    """Evaluate the quality of a response against expected criteria."""
    evaluation_prompt = f"""
    Evaluate the following response against the criteria: {expected_criteria}
    
    Response to evaluate: {response}
    
    Provide a score from 1-10 and brief explanation.
    Return as JSON: {{"score": int, "explanation": str}}
    """
    
    eval_result = eval_model.invoke(evaluation_prompt)
    try:
        return json.loads(eval_result.content)
    except:
        return {"score": 5, "explanation": "Unable to parse evaluation"}

@tool
def evaluate_factual_accuracy(response: str, reference_facts: List[str]) -> Dict[str, Any]:
    """Evaluate the factual accuracy of a response against reference facts."""
    facts_str = "\n".join(reference_facts)
    evaluation_prompt = f"""
    Check if the following response is factually accurate based on these reference facts:
    
    Reference Facts:
    {facts_str}
    
    Response to check: {response}
    
    Return JSON: {{"is_accurate": bool, "accuracy_score": int (1-10), "issues": [str]}}
    """
    
    eval_result = eval_model.invoke(evaluation_prompt)
    try:
        return json.loads(eval_result.content)
    except:
        return {"is_accurate": False, "accuracy_score": 5, "issues": ["Unable to parse evaluation"]}

@tool
def evaluate_relevance(response: str, query: str) -> Dict[str, Any]:
    """Evaluate how relevant a response is to the original query."""
    evaluation_prompt = f"""
    Evaluate how relevant this response is to the original query:
    
    Query: {query}
    Response: {response}
    
    Return JSON: {{"relevance_score": int (1-10), "explanation": str}}
    """
    
    eval_result = eval_model.invoke(evaluation_prompt)
    try:
        return json.loads(eval_result.content)
    except:
        return {"relevance_score": 5, "explanation": "Unable to parse evaluation"}

# Define the agent state
class AgentState:
    def __init__(self):
        self.messages: List[Any] = []
        self.evaluations: Dict[str, Any] = {}
        self.current_response: str = ""
        self.evaluation_criteria: Dict[str, Any] = {}

def evaluate_agent(
    response: str,
    query: str,
    evaluation_criteria: Dict[str, Any],
    reference_facts: Optional[List[str]] = None
) -> Dict[str, Any]:
    """
    Evaluate an agent's response using multiple criteria with Phoenix tracking.
    
    Args:
        response: The agent's response to evaluate
        query: The original query/question
        evaluation_criteria: Dict with evaluation parameters
        reference_facts: Optional list of reference facts for accuracy checking
    
    Returns:
        Dict containing evaluation results
    """
    
    # Bind tools to the model
    tools = [evaluate_response_quality, evaluate_factual_accuracy, evaluate_relevance]
    model_with_tools = client.bind_tools(tools)
    
    # Create tool node for execution
    tool_node = ToolNode(tools)
    
    with using_attributes(
        user_id="evaluation_user",
        session_id="eval_session",
        # task_type="agent_evaluation"

        
    ):
        with using_tags(["evaluation", "agent_assessment"]):
            evaluation_results = {}
            
            # Evaluate response quality
            if "quality_criteria" in evaluation_criteria:
                quality_eval = evaluate_response_quality.invoke({
                    "response": response,
                    "expected_criteria": evaluation_criteria["quality_criteria"]
                })
                evaluation_results["quality"] = quality_eval
            
            # Evaluate factual accuracy if reference facts provided
            if reference_facts:
                accuracy_eval = evaluate_factual_accuracy.invoke({
                    "response": response,
                    "reference_facts": reference_facts
                })
                evaluation_results["accuracy"] = accuracy_eval
            
            # Evaluate relevance
            relevance_eval = evaluate_relevance.invoke({
                "response": response,
                "query": query
            })
            evaluation_results["relevance"] = relevance_eval
            
            # Generate overall assessment using the model
            assessment_messages = [
                SystemMessage(content="""You are an expert evaluator. Based on the evaluation results provided, 
                give an overall assessment of the agent's response performance."""),
                HumanMessage(content=f"""
                Original Query: {query}
                Agent Response: {response}
                Evaluation Results: {json.dumps(evaluation_results, indent=2)}
                
                Provide an overall assessment with:
                1. Overall score (1-10)
                2. Key strengths
                3. Areas for improvement
                4. Recommendation
                """)
            ]
            
            overall_assessment = model_with_tools.invoke(assessment_messages)
            evaluation_results["overall_assessment"] = overall_assessment.content
            
            # Calculate composite score
            scores = []
            if "quality" in evaluation_results:
                scores.append(evaluation_results["quality"].get("score", 5))
            if "accuracy" in evaluation_results:
                scores.append(evaluation_results["accuracy"].get("accuracy_score", 5))
            if "relevance" in evaluation_results:
                scores.append(evaluation_results["relevance"].get("relevance_score", 5))
            
            composite_score = sum(scores) / len(scores) if scores else 5
            evaluation_results["composite_score"] = composite_score
            
            return evaluation_results

# Example usage function
def run_evaluation_example():
    """Example of how to use the evaluate_agent function."""
    
    # Sample agent response to evaluate
    sample_response = """
    The capital of France is Paris. Paris is located in the north-central part of France 
    and has been the country's capital since 1871. It's known for landmarks like the 
    Eiffel Tower and the Louvre Museum.
    """
    
    sample_query = "What is the capital of France and what is it known for?"
    
    # Define evaluation criteria
    evaluation_criteria = {
        "quality_criteria": "Response should be accurate, complete, and well-structured",
        "expected_format": "Should include the capital name and notable features"
    }
    
    # Reference facts for accuracy checking
    reference_facts = [
        "Paris is the capital of France",
        "Paris is located in north-central France",
        "The Eiffel Tower is located in Paris",
        "The Louvre Museum is located in Paris"
    ]
    
    # Run evaluation
    results = evaluate_agent(
        response=sample_response,
        query=sample_query,
        evaluation_criteria=evaluation_criteria,
        reference_facts=reference_facts
    )
    
    print("Evaluation Results:")
    print(json.dumps(results, indent=2))

if __name__ == "__main__":
    run_evaluation_example()


Overriding of current TracerProvider is not allowed


DependencyConflict: requested: "openai >= 1.69.0" but found: "None"
Attempting to instrument while already instrumented
Attempting to instrument while already instrumented


🔭 OpenTelemetry Tracing Details 🔭
|  Phoenix Project: evaluate_agent_project
|  Span Processor: SimpleSpanProcessor
|  Collector Endpoint: localhost:4317
|  Transport: gRPC
|  Transport Headers: {'user-agent': '****'}
|  
|  Using a default SpanProcessor. `add_span_processor` will overwrite this default.
|  
|  ⚠️ WARNING: It is strongly advised to use a BatchSpanProcessor in production environments.
|  
|  `register` has set this TracerProvider as the global OpenTelemetry default.
|  To disable this behavior, call `register` with `set_global_tracer_provider=False`.



This model can reply with multiple function calls in one response. Please don't rely on `additional_kwargs.function_call` as only the last one will be saved.Use `tool_calls` instead.
This model can reply with multiple function calls in one response. Please don't rely on `additional_kwargs.function_call` as only the last one will be saved.Use `tool_calls` instead.


Evaluation Results:
{
  "quality": {
    "score": 5,
    "explanation": "Unable to parse evaluation"
  },
  "accuracy": {
    "is_accurate": false,
    "accuracy_score": 5,
    "issues": [
      "Unable to parse evaluation"
    ]
  },
  "relevance": {
    "relevance_score": 5,
    "explanation": "Unable to parse evaluation"
  },
  "overall_assessment": "",
  "composite_score": 5.0
}


## Evaluate experiemnt  - custom

In [29]:
import os
import uuid
import pandas as pd
from typing import List, Dict, Any, Optional, Set
from langchain_google_vertexai import ChatVertexAI
from langchain_core.messages import HumanMessage, AIMessage, SystemMessage
from langchain_core.tools import tool
from langgraph.prebuilt import ToolNode
from langgraph.graph import StateGraph, END, START
from langgraph.graph.message import add_messages
from typing_extensions import Annotated, TypedDict
from phoenix.otel import register
from openinference.instrumentation.langchain import LangChainInstrumentor
from openinference.instrumentation import using_attributes, using_tags
import json
import phoenix as px

# Phoenix configuration
project_name = "tool_calling_evaluation"

os.environ["PHOENIX_COLLECTOR_ENDPOINT"] = "http://localhost:6006"
os.environ["PHOENIX_PROJECT_NAME"] = project_name

# Configure the Phoenix tracer
tracer_provider = register(
    project_name=project_name,
    auto_instrument=True
)

LangChainInstrumentor().instrument(tracer_provider=tracer_provider)
tracer = tracer_provider.get_tracer(__name__)

# Initialize ChatVertexAI model
model = ChatVertexAI(
    model_name="gemini-1.5-pro",
    temperature=0.1
)

# Initialize Phoenix client
px_client = px.Client()

# Define actual tools for the sales agent
@tool
def lookup_sales_data(query: str) -> str:
    """Look up sales data from the database based on a query."""
    # Simulate database lookup
    return f"Sales data retrieved for query: {query}"

@tool
def analyze_sales_data(data: str) -> str:
    """Analyze sales data and provide insights."""
    # Simulate data analysis
    return f"Analysis completed on: {data}"

@tool
def generate_visualization(chart_type: str, data: str) -> str:
    """Generate a visualization based on chart type and data."""
    # Simulate visualization generation
    return f"Generated {chart_type} visualization for: {data}"

@tool
def run_python_code(code: str) -> str:
    """Execute Python code for data processing."""
    # Simulate code execution
    return f"Executed Python code: {code}"

# Define the agent state
class AgentState(TypedDict):
    messages: Annotated[list, add_messages]
    tool_calls: List[str]

# Create the sales agent
def create_sales_agent():
    """Create a tool-calling sales agent using LangGraph."""
    
    # Available tools
    tools = [lookup_sales_data, analyze_sales_data, generate_visualization, run_python_code]
    tool_node = ToolNode(tools)
    
    # Bind tools to model
    model_with_tools = model.bind_tools(tools)
    
    def should_continue(state: AgentState) -> str:
        """Determine if we should continue or end."""
        messages = state["messages"]
        last_message = messages[-1]
        
        # If the LLM makes a tool call, then we route to the "tools" node
        if last_message.tool_calls:
            return "tools"
        # Otherwise, we stop (reply to the user)
        return END
    
    def call_model(state: AgentState) -> AgentState:
        """Call the model and track tool calls."""
        messages = state["messages"]
        response = model_with_tools.invoke(messages)
        
        # Track tool calls
        tool_calls = state.get("tool_calls", [])
        if response.tool_calls:
            for tool_call in response.tool_calls:
                tool_calls.append(tool_call["name"])
        
        return {"messages": [response], "tool_calls": tool_calls}
    
    def call_tools(state: AgentState) -> AgentState:
        """Call tools and return results."""
        messages = state["messages"]
        last_message = messages[-1]
        
        # Execute tools
        tool_outputs = tool_node.invoke({"messages": [last_message]})
        
        return {"messages": tool_outputs["messages"]}
    
    # Define the graph
    workflow = StateGraph(AgentState)
    
    # Add nodes
    workflow.add_node("agent", call_model)
    workflow.add_node("tools", call_tools)
    
    # Set entry point
    workflow.add_edge(START, "agent")
    
    # Add conditional edges
    workflow.add_conditional_edges(
        "agent",
        should_continue,
        {
            "tools": "tools",
            END: END,
        },
    )
    
    # Add edge from tools back to agent
    workflow.add_edge("tools", "agent")
    
    # Compile the graph
    app = workflow.compile()
    
    return app

# Tool calling evaluation functions
@tool
def evaluate_tool_accuracy(expected_tools: str, actual_tools: str) -> Dict[str, Any]:
    """Evaluate exact match accuracy between expected and actual tool calls."""
    expected_set = set(tool.strip() for tool in expected_tools.split(','))
    actual_set = set(tool.strip() for tool in actual_tools.split(','))
    
    exact_match = expected_set == actual_set
    accuracy_score = 10 if exact_match else 0
    
    return {
        "exact_match": exact_match,
        "accuracy_score": accuracy_score,
        "expected_tools": list(expected_set),
        "actual_tools": list(actual_set)
    }

@tool
def evaluate_tool_precision_recall(expected_tools: str, actual_tools: str) -> Dict[str, Any]:
    """Calculate precision and recall for tool calling."""
    expected_set = set(tool.strip() for tool in expected_tools.split(','))
    actual_set = set(tool.strip() for tool in actual_tools.split(','))
    
    if not actual_set:
        precision = 0.0
    else:
        precision = len(expected_set & actual_set) / len(actual_set)
    
    if not expected_set:
        recall = 1.0 if not actual_set else 0.0
    else:
        recall = len(expected_set & actual_set) / len(expected_set)
    
    f1 = 2 * (precision * recall) / (precision + recall) if (precision + recall) > 0 else 0.0
    
    return {
        "precision": precision,
        "recall": recall,
        "f1_score": f1,
        "true_positives": list(expected_set & actual_set),
        "false_positives": list(actual_set - expected_set),
        "false_negatives": list(expected_set - actual_set)
    }

@tool
def evaluate_tool_sequence(expected_tools: str, actual_tools: str) -> Dict[str, Any]:
    """Evaluate if tool calling sequence is appropriate."""
    expected_list = [tool.strip() for tool in expected_tools.split(',')]
    actual_list = [tool.strip() for tool in actual_tools.split(',')]
    
    # Check if the sequence makes logical sense
    evaluation_prompt = f"""
    Evaluate if this tool calling sequence is logical and appropriate:
    
    Expected sequence: {expected_list}
    Actual sequence: {actual_list}
    
    Consider:
    1. Are the tools called in a logical order?
    2. Are there any missing dependencies?
    3. Are there unnecessary tool calls?
    
    Return JSON: {{"sequence_score": int (1-10), "is_logical": bool, "issues": [str]}}
    """
    
    eval_result = model.invoke([HumanMessage(content=evaluation_prompt)])
    try:
        result = json.loads(eval_result.content)
        result["expected_sequence"] = expected_list
        result["actual_sequence"] = actual_list
        return result
    except:
        return {
            "sequence_score": 5,
            "is_logical": False,
            "issues": ["Unable to parse evaluation"],
            "expected_sequence": expected_list,
            "actual_sequence": actual_list
        }

def run_agent_on_question(agent_app, question: str) -> List[str]:
    """Run the agent on a question and return the tool calls made."""
    
    with using_attributes(
        user_id="agent_user",
        session_id="agent_session"
    ):
        with using_tags(["agent_execution", "tool_calling"]):
            
            # Create initial state
            initial_state = {
                "messages": [HumanMessage(content=question)],
                "tool_calls": []
            }
            
            # Run the agent
            final_state = agent_app.invoke(initial_state)
            
            # Return the tool calls made
            return final_state.get("tool_calls", [])

def run_experiment(dataset_df: pd.DataFrame, experiment_name: str = "tool_calling_experiment") -> Dict[str, Any]:
    """
    Run the tool calling experiment on the dataset.
    
    Args:
        dataset_df: DataFrame with 'question' and 'tool_calls' columns
        experiment_name: Name for the experiment
    
    Returns:
        Dict containing agent responses for each question
    """
    
    # Create the sales agent
    agent_app = create_sales_agent()
    
    agent_responses = {}
    
    with using_attributes(
        user_id="experiment_user",
        session_id=f"exp_{experiment_name}"
    ):
        with using_tags(["experiment", "tool_calling", "dataset"]):
            
            for index, row in dataset_df.iterrows():
                question = row['question']
                
                print(f"Processing question {index + 1}/{len(dataset_df)}: {question}")
                
                # Run agent on question
                tool_calls = run_agent_on_question(agent_app, question)
                
                # Convert tool calls list to comma-separated string
                tool_calls_str = ", ".join(tool_calls) if tool_calls else ""
                agent_responses[question] = tool_calls_str
                
                print(f"Agent tool calls: {tool_calls_str}")
    
    return agent_responses

def evaluate_experiment(
    dataset_df: pd.DataFrame,
    agent_responses: Dict[str, str],
    evaluation_name: str = "tool_calling_evaluation"
) -> Dict[str, Any]:
    """
    Evaluate the experiment results.
    
    Args:
        dataset_df: DataFrame with 'question' and 'tool_calls' columns
        agent_responses: Dict mapping questions to actual tool calls
        evaluation_name: Name for the evaluation run
    
    Returns:
        Dict containing comprehensive evaluation results
    """
    
    evaluation_results = {
        "evaluation_name": evaluation_name,
        "total_questions": len(dataset_df),
        "individual_results": [],
        "aggregate_metrics": {}
    }
    
    all_accuracy_scores = []
    all_precision_scores = []
    all_recall_scores = []
    all_f1_scores = []
    all_sequence_scores = []
    
    with using_attributes(
        user_id="evaluation_user",
        session_id=f"eval_{evaluation_name}"
    ):
        with using_tags(["evaluation", "tool_calling", "dataset"]):
            
            for index, row in dataset_df.iterrows():
                question = row['question']
                expected_tools = row['tool_calls']
                actual_tools = agent_responses.get(question, "")
                
                # Individual question evaluation
                question_results = {
                    "question": question,
                    "expected_tools": expected_tools,
                    "actual_tools": actual_tools
                }
                
                # Evaluate accuracy
                accuracy_result = evaluate_tool_accuracy.invoke({
                    "expected_tools": expected_tools,
                    "actual_tools": actual_tools
                })
                question_results["accuracy"] = accuracy_result
                all_accuracy_scores.append(accuracy_result["accuracy_score"])
                
                # Evaluate precision/recall
                precision_recall_result = evaluate_tool_precision_recall.invoke({
                    "expected_tools": expected_tools,
                    "actual_tools": actual_tools
                })
                question_results["precision_recall"] = precision_recall_result
                all_precision_scores.append(precision_recall_result["precision"])
                all_recall_scores.append(precision_recall_result["recall"])
                all_f1_scores.append(precision_recall_result["f1_score"])
                
                # Evaluate sequence
                sequence_result = evaluate_tool_sequence.invoke({
                    "expected_tools": expected_tools,
                    "actual_tools": actual_tools
                })
                question_results["sequence"] = sequence_result
                all_sequence_scores.append(sequence_result["sequence_score"])
                
                evaluation_results["individual_results"].append(question_results)
            
            # Calculate aggregate metrics
            evaluation_results["aggregate_metrics"] = {
                "overall_accuracy": sum(all_accuracy_scores) / len(all_accuracy_scores),
                "exact_match_rate": sum(1 for score in all_accuracy_scores if score == 10) / len(all_accuracy_scores),
                "average_precision": sum(all_precision_scores) / len(all_precision_scores),
                "average_recall": sum(all_recall_scores) / len(all_recall_scores),
                "average_f1": sum(all_f1_scores) / len(all_f1_scores),
                "average_sequence_score": sum(all_sequence_scores) / len(all_sequence_scores)
            }
            
            # Generate overall assessment
            assessment_messages = [
                SystemMessage(content="""You are an expert evaluator of AI agent tool calling performance. 
                Provide insights and recommendations based on the evaluation results."""),
                HumanMessage(content=f"""
                Tool Calling Evaluation Results:
                Total Questions: {evaluation_results['total_questions']}
                Exact Match Rate: {evaluation_results['aggregate_metrics']['exact_match_rate']:.2%}
                Average Precision: {evaluation_results['aggregate_metrics']['average_precision']:.3f}
                Average Recall: {evaluation_results['aggregate_metrics']['average_recall']:.3f}
                Average F1: {evaluation_results['aggregate_metrics']['average_f1']:.3f}
                Average Sequence Score: {evaluation_results['aggregate_metrics']['average_sequence_score']:.1f}/10
                
                Provide analysis with:
                1. Overall performance assessment
                2. Key strengths and weaknesses
                3. Specific recommendations for improvement
                4. Tool calling patterns observed
                """)
            ]
            
            overall_assessment = model.invoke(assessment_messages)
            evaluation_results["overall_assessment"] = overall_assessment.content
            
            return evaluation_results

def setup_ground_truth_dataset():
    """Setup the ground truth dataset as provided by the user."""
    
    id = str(uuid.uuid4())
    
    agent_tool_responses = {
        "What was the most popular product SKU?": "lookup_sales_data, analyze_sales_data",
        "What was the total revenue across all stores?": "lookup_sales_data, analyze_sales_data",
        "Which store had the highest sales volume?": "lookup_sales_data, analyze_sales_data",
        "Create a bar chart showing total sales by store": "generate_visualization, lookup_sales_data, run_python_code",
        "What percentage of items were sold on promotion?": "lookup_sales_data, analyze_sales_data",
        "Plot daily sales volume over time": "generate_visualization, lookup_sales_data, run_python_code",
        "What was the average transaction value?": "lookup_sales_data, analyze_sales_data",
        "Create a box plot of transaction values": "generate_visualization, lookup_sales_data, run_python_code",
        "Which products were frequently purchased together?": "lookup_sales_data, analyze_sales_data",
        "Plot a line graph showing the sales trend over time with a 7-day moving average": "generate_visualization, lookup_sales_data, run_python_code",
    }
    
    tool_calling_df = pd.DataFrame(agent_tool_responses.items(), columns=["question", "tool_calls"])
    
    dataset = px_client.upload_dataset(
        dataframe=tool_calling_df,
        dataset_name=f"tool_calling_ground_truth_{id}",
        input_keys=["question"],
        output_keys=["tool_calls"],
    )
    
    return tool_calling_df, dataset

def main():
    """Main function to run the complete tool calling evaluation."""
    
    print("=== Setting up Ground Truth Dataset ===")
    ground_truth_df, dataset = setup_ground_truth_dataset()
    print(f"Dataset created with {len(ground_truth_df)} questions")
    
    print("\n=== Running Experiment ===")
    agent_responses = run_experiment(
        dataset_df=ground_truth_df,
        experiment_name="sales_agent_tool_calling_v1"
    )
    
    print("\n=== Evaluating Results ===")
    evaluation_results = evaluate_experiment(
        dataset_df=ground_truth_df,
        agent_responses=agent_responses,
        evaluation_name="sales_agent_evaluation_v1"
    )
    
    # Print results
    print("\n=== Tool Calling Evaluation Results ===")
    print(f"Total Questions: {evaluation_results['total_questions']}")
    print(f"Exact Match Rate: {evaluation_results['aggregate_metrics']['exact_match_rate']:.2%}")
    print(f"Average Precision: {evaluation_results['aggregate_metrics']['average_precision']:.3f}")
    print(f"Average Recall: {evaluation_results['aggregate_metrics']['average_recall']:.3f}")
    print(f"Average F1 Score: {evaluation_results['aggregate_metrics']['average_f1']:.3f}")
    print(f"Average Sequence Score: {evaluation_results['aggregate_metrics']['average_sequence_score']:.1f}/10")
    
    print("\n=== Sample Individual Results ===")
    for i, result in enumerate(evaluation_results['individual_results'][:3]):
        print(f"\n{i+1}. Question: {result['question']}")
        print(f"   Expected: {result['expected_tools']}")
        print(f"   Actual: {result['actual_tools']}")
        print(f"   Exact Match: {result['accuracy']['exact_match']}")
        print(f"   Precision: {result['precision_recall']['precision']:.3f}")
        print(f"   Recall: {result['precision_recall']['recall']:.3f}")
    
    print(f"\n=== Overall Assessment ===")
    print(evaluation_results['overall_assessment'])
    
    return evaluation_results

if __name__ == "__main__":
    results = main()


Overriding of current TracerProvider is not allowed
DependencyConflict: requested: "openai >= 1.69.0" but found: "None"
Attempting to instrument while already instrumented
Attempting to instrument while already instrumented
/Users/stepan.lavrinenko/phoenix/.venv/lib/python3.12/site-packages/phoenix/utilities/client.py:51: UserWarning: ⚠️⚠️ The Phoenix server (10.13.2) and client (11.5.0) versions are severely mismatched. Upgrade  either the client or server to ensure API compatibility ⚠️⚠️
  warnings.warn(
/Users/stepan.lavrinenko/phoenix/.venv/lib/python3.12/site-packages/phoenix/utilities/client.py:51: UserWarning: ⚠️⚠️ The Phoenix server (10.13.2) and client (11.5.0) versions are severely mismatched. Upgrade  either the client or server to ensure API compatibility ⚠️⚠️
  warnings.warn(


🔭 OpenTelemetry Tracing Details 🔭
|  Phoenix Project: tool_calling_evaluation
|  Span Processor: SimpleSpanProcessor
|  Collector Endpoint: localhost:4317
|  Transport: gRPC
|  Transport Headers: {'user-agent': '****'}
|  
|  Using a default SpanProcessor. `add_span_processor` will overwrite this default.
|  
|  ⚠️ WARNING: It is strongly advised to use a BatchSpanProcessor in production environments.
|  
|  `register` has set this TracerProvider as the global OpenTelemetry default.
|  To disable this behavior, call `register` with `set_global_tracer_provider=False`.

=== Setting up Ground Truth Dataset ===
📤 Uploading dataset...
💾 Examples uploaded: http://localhost:6006/datasets/RGF0YXNldDo3/examples
🗄️ Dataset version ID: RGF0YXNldFZlcnNpb246Ng==
Dataset created with 10 questions

=== Running Experiment ===
Processing question 1/10: What was the most popular product SKU?
Agent tool calls: lookup_sales_data
Processing question 2/10: What was the total revenue across all stores?
Agent

## Evaluate experiment - phoenix

In [31]:
import os
import uuid
import pandas as pd
from typing import List, Dict, Any, Optional, Set
from langchain_google_vertexai import ChatVertexAI
from langchain_core.messages import HumanMessage, AIMessage, SystemMessage
from langchain_core.tools import tool
from langgraph.prebuilt import ToolNode
from langgraph.graph import StateGraph, END, START
from langgraph.graph.message import add_messages
from typing_extensions import Annotated, TypedDict
from phoenix.otel import register
from openinference.instrumentation.langchain import LangChainInstrumentor
from openinference.instrumentation import using_attributes, using_tags
from phoenix.experiments import evaluate_experiment, run_experiment
import json
import phoenix as px

# Phoenix configuration
project_name = "tool_calling_evaluation"

os.environ["PHOENIX_COLLECTOR_ENDPOINT"] = "http://localhost:6006"
os.environ["PHOENIX_PROJECT_NAME"] = project_name

# Configure the Phoenix tracer
tracer_provider = register(
    project_name=project_name,
    auto_instrument=True
)

LangChainInstrumentor().instrument(tracer_provider=tracer_provider)
tracer = tracer_provider.get_tracer(__name__)

# Initialize ChatVertexAI model
model = ChatVertexAI(
    model_name="gemini-1.5-pro",
    temperature=0.1
)

# Initialize Phoenix client
px_client = px.Client()

# Define actual tools for the sales agent
@tool
def lookup_sales_data(query: str) -> str:
    """Look up sales data from the database based on a query."""
    return f"Sales data retrieved for query: {query}"

@tool
def analyze_sales_data(data: str) -> str:
    """Analyze sales data and provide insights."""
    return f"Analysis completed on: {data}"

@tool
def generate_visualization(chart_type: str, data: str) -> str:
    """Generate a visualization based on chart type and data."""
    return f"Generated {chart_type} visualization for: {data}"

@tool
def run_python_code(code: str) -> str:
    """Execute Python code for data processing."""
    return f"Executed Python code: {code}"

# Define the agent state
class AgentState(TypedDict):
    messages: Annotated[list, add_messages]
    tool_calls: List[str]

# Create the sales agent
def create_sales_agent():
    """Create a tool-calling sales agent using LangGraph."""
    
    # Available tools
    tools = [lookup_sales_data, analyze_sales_data, generate_visualization, run_python_code]
    tool_node = ToolNode(tools)
    
    # Bind tools to model
    model_with_tools = model.bind_tools(tools)
    
    def should_continue(state: AgentState) -> str:
        """Determine if we should continue or end."""
        messages = state["messages"]
        last_message = messages[-1]
        
        # If the LLM makes a tool call, then we route to the "tools" node
        if last_message.tool_calls:
            return "tools"
        # Otherwise, we stop (reply to the user)
        return END
    
    def call_model(state: AgentState) -> AgentState:
        """Call the model and track tool calls."""
        messages = state["messages"]
        response = model_with_tools.invoke(messages)
        
        # Track tool calls
        tool_calls = state.get("tool_calls", [])
        if response.tool_calls:
            for tool_call in response.tool_calls:
                tool_calls.append(tool_call["name"])
        
        return {"messages": [response], "tool_calls": tool_calls}
    
    def call_tools(state: AgentState) -> AgentState:
        """Call tools and return results."""
        messages = state["messages"]
        last_message = messages[-1]
        
        # Execute tools
        tool_outputs = tool_node.invoke({"messages": [last_message]})
        
        return {"messages": tool_outputs["messages"]}
    
    # Define the graph
    workflow = StateGraph(AgentState)
    
    # Add nodes
    workflow.add_node("agent", call_model)
    workflow.add_node("tools", call_tools)
    
    # Set entry point
    workflow.add_edge(START, "agent")
    
    # Add conditional edges
    workflow.add_conditional_edges(
        "agent",
        should_continue,
        {
            "tools": "tools",
            END: END,
        },
    )
    
    # Add edge from tools back to agent
    workflow.add_edge("tools", "agent")
    
    # Compile the graph
    app = workflow.compile()
    
    return app

# Task function for Phoenix experiment
def sales_agent_task(input_data: Dict[str, Any]) -> Dict[str, Any]:
    """
    Task function that runs the sales agent on a question.
    This function will be used by Phoenix's run_experiment.
    
    Args:
        input_data: Dict containing the question and expected tool calls
        
    Returns:
        Dict containing the agent's response and tool calls
    """
    question = input_data["question"]
    
    # Create the sales agent
    agent_app = create_sales_agent()
    
    with using_attributes(
        user_id="experiment_user",
        session_id="phoenix_experiment"
    ):
        with using_tags(["phoenix_experiment", "tool_calling"]):
            
            # Create initial state
            initial_state = {
                "messages": [HumanMessage(content=question)],
                "tool_calls": []
            }
            
            # Run the agent
            final_state = agent_app.invoke(initial_state)
            
            # Get the final response
            final_message = final_state["messages"][-1]
            response_content = final_message.content if hasattr(final_message, 'content') else str(final_message)
            
            # Get tool calls
            tool_calls = final_state.get("tool_calls", [])
            tool_calls_str = ", ".join(tool_calls) if tool_calls else ""
            
            return {
                "response": response_content,
                "tool_calls": tool_calls_str,
                "individual_tool_calls": tool_calls
            }

# Evaluation functions for Phoenix
def tool_accuracy_evaluator(output: Dict[str, Any], expected: Dict[str, Any]) -> float:
    """Evaluate tool calling accuracy - returns a score between 0 and 1."""
    expected_tools = expected.get("tool_calls", "")
    actual_tools = output.get("tool_calls", "")
    
    expected_set = set(tool.strip() for tool in expected_tools.split(',') if tool.strip())
    actual_set = set(tool.strip() for tool in actual_tools.split(',') if tool.strip())
    
    exact_match = expected_set == actual_set
    return 1.0 if exact_match else 0.0

def tool_precision_recall_evaluator(output: Dict[str, Any], expected: Dict[str, Any]) -> float:
    """Evaluate tool calling precision and recall - returns F1 score."""
    expected_tools = expected.get("tool_calls", "")
    actual_tools = output.get("tool_calls", "")
    
    expected_set = set(tool.strip() for tool in expected_tools.split(',') if tool.strip())
    actual_set = set(tool.strip() for tool in actual_tools.split(',') if tool.strip())
    
    if not actual_set:
        precision = 0.0
    else:
        precision = len(expected_set & actual_set) / len(actual_set)
    
    if not expected_set:
        recall = 1.0 if not actual_set else 0.0
    else:
        recall = len(expected_set & actual_set) / len(expected_set)
    
    f1 = 2 * (precision * recall) / (precision + recall) if (precision + recall) > 0 else 0.0
    return f1

def tool_sequence_evaluator(output: Dict[str, Any], expected: Dict[str, Any]) -> float:
    """Evaluate tool calling sequence logic - returns a score between 0 and 1."""
    expected_tools = expected.get("tool_calls", "")
    actual_tools = output.get("tool_calls", "")
    
    expected_list = [tool.strip() for tool in expected_tools.split(',') if tool.strip()]
    actual_list = [tool.strip() for tool in actual_tools.split(',') if tool.strip()]
    
    # Simple sequence evaluation
    score = 0.5  # Base score
    
    # Check for logical patterns
    if "generate_visualization" in expected_list:
        if "generate_visualization" in actual_list:
            score += 0.3
        if "lookup_sales_data" in actual_list:
            score += 0.2
    else:
        if "lookup_sales_data" in actual_list and "analyze_sales_data" in actual_list:
            score += 0.5
    
    return min(score, 1.0)  # Cap at 1.0

def setup_ground_truth_dataset():
    """Setup the ground truth dataset for Phoenix experiments."""
    
    id = str(uuid.uuid4())
    
    agent_tool_responses = {
        "What was the most popular product SKU?": "lookup_sales_data, analyze_sales_data",
        "What was the total revenue across all stores?": "lookup_sales_data, analyze_sales_data",
        "Which store had the highest sales volume?": "lookup_sales_data, analyze_sales_data",
        "Create a bar chart showing total sales by store": "generate_visualization, lookup_sales_data, run_python_code",
        "What percentage of items were sold on promotion?": "lookup_sales_data, analyze_sales_data",
        "Plot daily sales volume over time": "generate_visualization, lookup_sales_data, run_python_code",
        "What was the average transaction value?": "lookup_sales_data, analyze_sales_data",
        "Create a box plot of transaction values": "generate_visualization, lookup_sales_data, run_python_code",
        "Which products were frequently purchased together?": "lookup_sales_data, analyze_sales_data",
        "Plot a line graph showing the sales trend over time with a 7-day moving average": "generate_visualization, lookup_sales_data, run_python_code",
    }
    
    # Prepare data for Phoenix experiment format
    experiment_data = []
    for question, expected_tools in agent_tool_responses.items():
        experiment_data.append({
            "question": question,
            "tool_calls": expected_tools
        })
    
    tool_calling_df = pd.DataFrame(experiment_data)
    
    # Upload to Phoenix
    dataset = px_client.upload_dataset(
        dataframe=tool_calling_df,
        dataset_name=f"tool_calling_ground_truth_{id}",
        input_keys=["question"],
        output_keys=["tool_calls"],
    )
    
    return tool_calling_df, dataset

def main():
    """Main function using Phoenix's experiment framework correctly."""
    
    print("=== Setting up Ground Truth Dataset ===")
    ground_truth_df, dataset = setup_ground_truth_dataset()
    print(f"Dataset created with {len(ground_truth_df)} questions")
    
    print("\n=== Running Phoenix Experiment ===")
    
    # CORRECT USAGE: Run experiment with evaluators included
    experiment = run_experiment(
        dataset,
        sales_agent_task,
        evaluators=[tool_accuracy_evaluator],  # Include initial evaluators
        experiment_name="Tool Calling Eval",
        experiment_description="Evaluating the tool calling step of the agent",
    )
    
    print(f"Initial experiment completed")
    
    print("\n=== Adding Additional Evaluators ===")
    
    # CORRECT USAGE: Add additional evaluators (no experiment_name parameter)
    experiment = evaluate_experiment(
        experiment, 
        evaluators=[tool_precision_recall_evaluator, tool_sequence_evaluator]
    )
    
    print("\n=== Phoenix Evaluation Results ===")
    
    # Extract and display results
    if hasattr(experiment, 'get_results'):
        results_df = experiment.get_results()
    elif hasattr(experiment, 'dataframe'):
        results_df = experiment.dataframe
    else:
        print(f"Experiment type: {type(experiment)}")
        print(f"Available methods: {[method for method in dir(experiment) if not method.startswith('_')]}")
        return experiment
    
    # Calculate aggregate metrics
    print(f"Total Questions: {len(results_df)}")
    
    # Display available columns for debugging
    print(f"Available columns: {list(results_df.columns)}")
    
    # Try to extract scores (column names may vary)
    accuracy_col = None
    precision_recall_col = None
    sequence_col = None
    
    for col in results_df.columns:
        if 'accuracy' in col.lower():
            accuracy_col = col
        elif 'precision' in col.lower() or 'recall' in col.lower():
            precision_recall_col = col
        elif 'sequence' in col.lower():
            sequence_col = col
    
    if accuracy_col:
        accuracy_scores = results_df[accuracy_col].dropna()
        print(f"Average Tool Accuracy: {accuracy_scores.mean():.3f}")
        print(f"Exact Match Rate: {(accuracy_scores == 1.0).mean():.2%}")
    
    if precision_recall_col:
        precision_recall_scores = results_df[precision_recall_col].dropna()
        print(f"Average Precision/Recall F1: {precision_recall_scores.mean():.3f}")
    
    if sequence_col:
        sequence_scores = results_df[sequence_col].dropna()
        print(f"Average Sequence Score: {sequence_scores.mean():.3f}")
    
    print("\n=== Sample Results ===")
    for i in range(min(3, len(results_df))):
        row = results_df.iloc[i]
        print(f"\n{i+1}. Available data for row {i}:")
        for col in results_df.columns:
            if not col.startswith('_'):  # Skip internal columns
                print(f"   {col}: {row.get(col, 'N/A')}")
    
    return experiment

if __name__ == "__main__":
    results = main()




Overriding of current TracerProvider is not allowed
DependencyConflict: requested: "openai >= 1.69.0" but found: "None"
Attempting to instrument while already instrumented
Attempting to instrument while already instrumented
/Users/stepan.lavrinenko/phoenix/.venv/lib/python3.12/site-packages/phoenix/utilities/client.py:51: UserWarning: ⚠️⚠️ The Phoenix server (10.13.2) and client (11.5.0) versions are severely mismatched. Upgrade  either the client or server to ensure API compatibility ⚠️⚠️
  warnings.warn(
/Users/stepan.lavrinenko/phoenix/.venv/lib/python3.12/site-packages/phoenix/utilities/client.py:51: UserWarning: ⚠️⚠️ The Phoenix server (10.13.2) and client (11.5.0) versions are severely mismatched. Upgrade  either the client or server to ensure API compatibility ⚠️⚠️
  warnings.warn(
🐌!! If running inside a notebook, patching the event loop with nest_asyncio will allow asynchronous eval submission, and is significantly faster. To patch the event loop, run `nest_asyncio.apply()`.


🔭 OpenTelemetry Tracing Details 🔭
|  Phoenix Project: tool_calling_evaluation
|  Span Processor: SimpleSpanProcessor
|  Collector Endpoint: localhost:4317
|  Transport: gRPC
|  Transport Headers: {'user-agent': '****'}
|  
|  Using a default SpanProcessor. `add_span_processor` will overwrite this default.
|  
|  ⚠️ WARNING: It is strongly advised to use a BatchSpanProcessor in production environments.
|  
|  `register` has set this TracerProvider as the global OpenTelemetry default.
|  To disable this behavior, call `register` with `set_global_tracer_provider=False`.

=== Setting up Ground Truth Dataset ===
📤 Uploading dataset...
💾 Examples uploaded: http://localhost:6006/datasets/RGF0YXNldDo5/examples
🗄️ Dataset version ID: RGF0YXNldFZlcnNpb246OA==
Dataset created with 10 questions

=== Running Phoenix Experiment ===
🧪 Experiment started.
📺 View dataset experiments: http://localhost:6006/datasets/RGF0YXNldDo5/experiments
🔗 View this experiment: http://localhost:6006/datasets/RGF0YXNld


/Users/stepan.lavrinenko/phoenix/.venv/lib/python3.12/site-packages/phoenix/utilities/client.py:51: UserWarning: ⚠️⚠️ The Phoenix server (10.13.2) and client (11.5.0) versions are severely mismatched. Upgrade  either the client or server to ensure API compatibility ⚠️⚠️
  warnings.warn(










running tasks |██████████| 10/10 (100.0%) | ⏳ 00:48<00:00 |  4.83s/it
🐌!! If running inside a notebook, patching the event loop with nest_asyncio will allow asynchronous eval submission, and is significantly faster. To patch the event loop, run `nest_asyncio.apply()`.


✅ Task runs completed.
🧠 Evaluation started.



/Users/stepan.lavrinenko/phoenix/.venv/lib/python3.12/site-packages/phoenix/utilities/client.py:51: UserWarning: ⚠️⚠️ The Phoenix server (10.13.2) and client (11.5.0) versions are severely mismatched. Upgrade  either the client or server to ensure API compatibility ⚠️⚠️
  warnings.warn(
running experiment evaluations |██████████| 10/10 (100.0%) | ⏳ 00:00<00:00 | 181.31it/s
🐌!! If running inside a notebook, patching the event loop with nest_asyncio will allow asynchronous eval submission, and is significantly faster. To patch the event loop, run `nest_asyncio.apply()`.



🔗 View this experiment: http://localhost:6006/datasets/RGF0YXNldDo5/compare?experimentId=RXhwZXJpbWVudDo0

Experiment Summary (07/09/25 04:42 PM -0600)
--------------------------------------------
                 evaluator   n  n_scores  avg_score
0  tool_accuracy_evaluator  10        10        0.1

Tasks Summary (07/09/25 04:42 PM -0600)
---------------------------------------
   n_examples  n_runs  n_errors
0          10      10         0
Initial experiment completed

=== Adding Additional Evaluators ===
🧠 Evaluation started.



/Users/stepan.lavrinenko/phoenix/.venv/lib/python3.12/site-packages/phoenix/utilities/client.py:51: UserWarning: ⚠️⚠️ The Phoenix server (10.13.2) and client (11.5.0) versions are severely mismatched. Upgrade  either the client or server to ensure API compatibility ⚠️⚠️
  warnings.warn(

running experiment evaluations |██████████| 20/20 (100.0%) | ⏳ 00:00<00:00 | 160.87it/s


🔗 View this experiment: http://localhost:6006/datasets/RGF0YXNldDo5/compare?experimentId=RXhwZXJpbWVudDo0

Experiment Summary (07/09/25 04:42 PM -0600)
--------------------------------------------
                         evaluator   n  n_scores  avg_score
0  tool_precision_recall_evaluator  10        10       0.66
1          tool_sequence_evaluator  10        10       0.65

Experiment Summary (07/09/25 04:42 PM -0600)
--------------------------------------------
                 evaluator   n  n_scores  avg_score
0  tool_accuracy_evaluator  10        10        0.1

Tasks Summary (07/09/25 04:42 PM -0600)
---------------------------------------
   n_examples  n_runs  n_errors
0          10      10         0

=== Phoenix Evaluation Results ===
Experiment type: <class 'phoenix.experiments.types.RanExperiment'>
Available methods: ['add', 'as_dataframe', 'dataset', 'dataset_id', 'dataset_version_id', 'eval_runs', 'eval_summaries', 'from_dict', 'get_evaluations', 'id', 'info', 'params', 'p

## Evaluate tool args and tool order

## Eval with phoenix experiments. simpler state, eval against ground truth data

In [ ]:
import os
import uuid
import pandas as pd
from typing import List, Dict, Any, Optional, Set, Tuple
from langchain_google_vertexai import ChatVertexAI
from langchain_core.messages import HumanMessage, AIMessage, SystemMessage
from langchain_core.messages.utils import filter_messages
from langchain_core.tools import tool
from langgraph.prebuilt import ToolNode
from langgraph.graph import StateGraph, END, START
from langgraph.graph.message import add_messages
from typing_extensions import Annotated, TypedDict
from phoenix.otel import register
from openinference.instrumentation.langchain import LangChainInstrumentor
from openinference.instrumentation import using_attributes, using_tags
from phoenix.experiments import evaluate_experiment, run_experiment
import json
import phoenix as px

# Phoenix configuration
project_name = "tool_calling_evaluation"

os.environ["PHOENIX_COLLECTOR_ENDPOINT"] = "http://localhost:6006"
os.environ["PHOENIX_PROJECT_NAME"] = project_name

# Configure the Phoenix tracer
tracer_provider = register(
    project_name=project_name,
    auto_instrument=True
)

LangChainInstrumentor().instrument(tracer_provider=tracer_provider)
from openinference.instrumentation import using_attributes, using_tags

tracer = tracer_provider.get_tracer(__name__)

# Initialize ChatVertexAI model
model = ChatVertexAI(
    model_name="gemini-1.5-pro",
    temperature=0.1
)

# Initialize Phoenix client
px_client = px.Client()

# Define actual tools for the sales agent
@tool
def lookup_sales_data(query: str) -> str:
    """Look up sales data from the database based on a query."""
    return f"Sales data retrieved for query: {query}"

@tool
def analyze_sales_data(data: str) -> str:
    """Analyze sales data and provide insights."""
    return f"Analysis completed on: {data}"

@tool
def generate_visualization(chart_type: str, data: str) -> str:
    """Generate a visualization based on chart type and data."""
    return f"Generated {chart_type} visualization for: {data}"

@tool
def run_python_code(code: str) -> str:
    """Execute Python code for data processing."""
    return f"Executed Python code: {code}"

# Simplified agent state - only messages
class AgentState(TypedDict):
    messages: Annotated[list, add_messages]

# Fixed helper function to extract tool information from messages using filter_messages
def extract_tool_calls_from_messages(messages: List[Any]) -> Dict[str, Any]:
    """Extract tool calls and arguments from message history using filter_messages."""
    tool_calls = []
    tool_call_details = []
    execution_order = []
    
    # Filter for AI messages that contain tool calls
    ai_messages = filter_messages(messages, include_types=[AIMessage])
    
    for message in ai_messages:
        # Check if this AI message has tool calls
        if hasattr(message, 'tool_calls') and message.tool_calls:
            for tool_call in message.tool_calls:
                tool_name = tool_call["name"]
                tool_args = tool_call.get("args", {})
                
                tool_calls.append(tool_name)
                execution_order.append(tool_name)
                
                tool_call_details.append({
                    "name": tool_name,
                    "args": tool_args,
                    "call_id": tool_call.get("id", ""),
                    "order": len(execution_order)
                })
        
        # Also check for additional_kwargs which might contain tool calls
        elif hasattr(message, 'additional_kwargs') and message.additional_kwargs:
            additional_kwargs = message.additional_kwargs
            if 'tool_calls' in additional_kwargs:
                for tool_call in additional_kwargs['tool_calls']:
                    # Handle different tool call formats
                    if isinstance(tool_call, dict):
                        if 'function' in tool_call:
                            # OpenAI-style format
                            tool_name = tool_call['function']['name']
                            try:
                                tool_args = json.loads(tool_call['function']['arguments'])
                            except:
                                tool_args = {}
                        else:
                            # Direct format
                            tool_name = tool_call.get('name', '')
                            tool_args = tool_call.get('args', {})
                        
                        if tool_name:
                            tool_calls.append(tool_name)
                            execution_order.append(tool_name)
                            
                            tool_call_details.append({
                                "name": tool_name,
                                "args": tool_args,
                                "call_id": tool_call.get("id", ""),
                                "order": len(execution_order)
                            })
    
    # Also check for tool messages that indicate tool execution
    from langchain_core.messages import ToolMessage
    tool_messages = filter_messages(messages, include_types=[ToolMessage])
    
    for tool_msg in tool_messages:
        if hasattr(tool_msg, 'name') and tool_msg.name:
            # This indicates a tool was actually executed
            tool_name = tool_msg.name
            
            # Check if we already have this tool call recorded
            if tool_name not in tool_calls:
                tool_calls.append(tool_name)
                execution_order.append(tool_name)
                
                tool_call_details.append({
                    "name": tool_name,
                    "args": {},  # Args not available in tool message
                    "call_id": getattr(tool_msg, 'tool_call_id', ''),
                    "order": len(execution_order)
                })
    
    return {
        "tool_calls": tool_calls,
        "tool_call_details": tool_call_details,
        "execution_order": execution_order
    }

# Create the sales agent with simplified state
def create_sales_agent():
    """Create a tool-calling sales agent using LangGraph with simplified state."""
    
    # Available tools
    tools = [lookup_sales_data, analyze_sales_data, generate_visualization, run_python_code]
    tool_node = ToolNode(tools)
    
    # Bind tools to model
    model_with_tools = model.bind_tools(tools)
    
    def should_continue(state: AgentState) -> str:
        """Determine if we should continue or end."""
        messages = state["messages"]
        last_message = messages[-1]
        
        # If the LLM makes a tool call, then we route to the "tools" node
        if hasattr(last_message, 'tool_calls') and last_message.tool_calls:
            return "tools"
        # Otherwise, we stop (reply to the user)
        return END
    
    def call_model(state: AgentState) -> AgentState:
        """Call the model and return response."""
        messages = state["messages"]
        response = model_with_tools.invoke(messages)
        return {"messages": [response]}
    
    def call_tools(state: AgentState) -> AgentState:
        """Call tools and return results."""
        messages = state["messages"]
        last_message = messages[-1]
        
        # Execute tools
        tool_outputs = tool_node.invoke({"messages": [last_message]})
        
        return {"messages": tool_outputs["messages"]}
    
    # Define the graph
    workflow = StateGraph(AgentState)
    
    # Add nodes
    workflow.add_node("agent", call_model)
    workflow.add_node("tools", call_tools)
    
    # Set entry point
    workflow.add_edge(START, "agent")
    
    # Add conditional edges
    workflow.add_conditional_edges(
        "agent",
        should_continue,
        {
            "tools": "tools",
            END: END,
        },
    )
    
    # Add edge from tools back to agent
    workflow.add_edge("tools", "agent")
    
    # Compile the graph
    app = workflow.compile()
    
    return app

# Simplified task function
def sales_agent_task(input_data: Dict[str, Any]) -> Dict[str, Any]:
    """Task function that runs the sales agent on a question."""
    question = input_data["question"]
    
    # Create the sales agent
    agent_app = create_sales_agent()
    
    with using_attributes(
        user_id="experiment_user",
        session_id="phoenix_experiment"
    ):
        with using_tags(["phoenix_experiment", "tool_calling"]):
            
            # Create initial state with only messages
            initial_state = {
                "messages": [HumanMessage(content=question)]
            }
            
            # Run the agent
            final_state = agent_app.invoke(initial_state)
            
            # Extract tool information from messages
            tool_info = extract_tool_calls_from_messages(final_state["messages"])
            
            # Get the final response
            final_message = final_state["messages"][-1]
            response_content = final_message.content if hasattr(final_message, 'content') else str(final_message)
            
            # Debug print to see what we extracted
            print(f"Question: {question}")
            print(f"Extracted tool calls: {tool_info['tool_calls']}")
            print(f"Tool call details: {tool_info['tool_call_details']}")
            print("---")
            
            return {
                "response": response_content,
                "messages": final_state["messages"],
                "tool_calls": ", ".join(tool_info["tool_calls"]) if tool_info["tool_calls"] else "",
                "individual_tool_calls": tool_info["tool_calls"],
                "tool_call_details": tool_info["tool_call_details"],
                "execution_order": tool_info["execution_order"]
            }

# Enhanced evaluation functions with flexible scoring

def tool_accuracy_evaluator(output: Dict[str, Any], expected: Dict[str, Any]) -> float:
    """Evaluate tool calling accuracy with flexible matching."""
    # Extract tool calls from messages
    messages = output.get("messages", [])
    tool_info = extract_tool_calls_from_messages(messages)
    actual_tools = set(tool_info["tool_calls"])
    
    # Get expected tools from ground truth - now supports multiple acceptable combinations
    expected_combinations = expected.get("expected_tool_combinations", [])
    
    # Check if actual tools match any of the acceptable combinations
    for combination in expected_combinations:
        expected_tools = set(tool["name"] for tool in combination)
        if expected_tools == actual_tools:
            return 1.0
        
        # Partial credit for subset matches
        if actual_tools.issubset(expected_tools) and len(actual_tools) > 0:
            return len(actual_tools) / len(expected_tools) * 0.8
    
    return 0.0

def tool_arguments_evaluator(output: Dict[str, Any], expected: Dict[str, Any]) -> float:
    """Evaluate if correct arguments were passed to tools based on ground truth."""
    # Extract tool calls from messages
    messages = output.get("messages", [])
    tool_info = extract_tool_calls_from_messages(messages)
    actual_tool_details = tool_info["tool_call_details"]
    
    if not actual_tool_details:
        return 0.0
    
    # More lenient argument evaluation
    total_score = 0.0
    total_tools = len(actual_tool_details)
    
    for actual_tool in actual_tool_details:
        tool_name = actual_tool["name"]
        tool_args = actual_tool.get("args", {})
        
        # Base score for having the tool
        tool_score = 0.5
        
        # Check arguments based on tool type
        if tool_name == "lookup_sales_data":
            query = tool_args.get("query", "")
            if query and len(query.strip()) > 3:
                tool_score += 0.5
        
        elif tool_name == "generate_visualization":
            chart_type = tool_args.get("chart_type", "")
            data = tool_args.get("data", "")
            if chart_type and len(chart_type.strip()) > 0:
                tool_score += 0.25
            if data and len(data.strip()) > 0:
                tool_score += 0.25
        
        elif tool_name == "run_python_code":
            code = tool_args.get("code", "")
            if code and len(code.strip()) > 5:
                tool_score += 0.5
        
        elif tool_name == "analyze_sales_data":
            data = tool_args.get("data", "")
            if data and len(data.strip()) > 0:
                tool_score += 0.5
        
        total_score += min(tool_score, 1.0)
    
    return total_score / total_tools if total_tools > 0 else 0.0

def tool_execution_order_evaluator(output: Dict[str, Any], expected: Dict[str, Any]) -> float:
    """Evaluate if tools were executed in a reasonable order."""
    # Extract execution order from messages
    messages = output.get("messages", [])
    tool_info = extract_tool_calls_from_messages(messages)
    actual_order = tool_info["execution_order"]
    
    if not actual_order:
        return 0.0
    
    # Basic order rules (more flexible)
    score = 1.0
    
    # Rule 1: lookup_sales_data should generally come before other tools
    if "lookup_sales_data" in actual_order:
        lookup_index = actual_order.index("lookup_sales_data")
        # Check if other tools come after lookup (with some tolerance)
        for tool in ["analyze_sales_data", "generate_visualization"]:
            if tool in actual_order:
                tool_index = actual_order.index(tool)
                if tool_index < lookup_index:
                    score *= 0.8  # Small penalty, not major
    
    # Rule 2: generate_visualization should come before run_python_code for complex viz
    if "generate_visualization" in actual_order and "run_python_code" in actual_order:
        viz_index = actual_order.index("generate_visualization")
        code_index = actual_order.index("run_python_code")
        if code_index < viz_index:
            score *= 0.9  # Small penalty
    
    return score

def tool_argument_values_evaluator(output: Dict[str, Any], expected: Dict[str, Any]) -> float:
    """Evaluate the quality of argument values with lenient criteria."""
    # Extract tool calls from messages
    messages = output.get("messages", [])
    tool_info = extract_tool_calls_from_messages(messages)
    actual_tool_details = tool_info["tool_call_details"]
    
    question = expected.get("question", "")
    
    if not actual_tool_details:
        return 0.0
    
    total_score = 0.0
    total_evaluations = 0
    
    for tool_detail in actual_tool_details:
        tool_name = tool_detail.get("name", "")
        tool_args = tool_detail.get("args", {})
        
        if tool_name == "lookup_sales_data":
            query = tool_args.get("query", "")
            score = evaluate_query_relevance(query, question)
            total_score += score
            total_evaluations += 1
        
        elif tool_name == "generate_visualization":
            chart_type = tool_args.get("chart_type", "")
            score = evaluate_chart_type_appropriateness(chart_type, question)
            total_score += score
            total_evaluations += 1
        
        elif tool_name == "run_python_code":
            code = tool_args.get("code", "")
            score = evaluate_code_relevance(code, question)
            total_score += score
            total_evaluations += 1
    
    return total_score / total_evaluations if total_evaluations > 0 else 0.0

def evaluate_query_relevance(query: str, question: str) -> float:
    """Evaluate how relevant the query is to the question - more lenient."""
    if not query:
        return 0.0
    
    # Base score for any reasonable query
    base_score = 0.7
    
    question_lower = question.lower()
    query_lower = str(query).lower()
    
    # Bonus for relevant terms
    bonus = 0.0
    relevant_terms = ["product", "sku", "revenue", "sales", "store", "transaction", "promotion"]
    
    for term in relevant_terms:
        if term in question_lower and term in query_lower:
            bonus += 0.05
    
    return min(1.0, base_score + bonus)

def evaluate_chart_type_appropriateness(chart_type: str, question: str) -> float:
    """Evaluate if the chart type is appropriate for the question - more lenient."""
    if not chart_type:
        return 0.0
    
    question_lower = question.lower()
    chart_type_lower = str(chart_type).lower()
    
    # Base score for any chart type
    base_score = 0.6
    
    # Bonus for appropriate types
    if "bar" in question_lower and "bar" in chart_type_lower:
        return 1.0
    elif "line" in question_lower and "line" in chart_type_lower:
        return 1.0
    elif "box" in question_lower and "box" in chart_type_lower:
        return 1.0
    elif any(word in question_lower for word in ["chart", "plot", "graph"]):
        return base_score + 0.2
    
    return base_score

def evaluate_code_relevance(code: str, question: str) -> float:
    """Evaluate if the code is relevant to the question - more lenient."""
    if not code:
        return 0.0
    
    # Base score for any code
    base_score = 0.6
    
    question_lower = question.lower()
    code_lower = str(code).lower()
    
    # Bonus for relevant operations
    bonus = 0.0
    if "average" in question_lower and any(term in code_lower for term in ["mean", "average", "avg"]):
        bonus += 0.2
    if "plot" in question_lower and any(term in code_lower for term in ["plot", "chart", "graph"]):
        bonus += 0.2
    
    return min(1.0, base_score + bonus)

def setup_realistic_ground_truth_dataset():
    """Setup realistic ground truth dataset with multiple acceptable tool combinations."""
    
    id = str(uuid.uuid4())
    
    # More realistic ground truth with multiple acceptable combinations
    realistic_ground_truth = [
        {
            "question": "What was the most popular product SKU?",
            "expected_tool_combinations": [
                # Option 1: Just lookup (LLM can analyze directly)
                [{"name": "lookup_sales_data", "args": {"query": "RELEVANT"}}],
                # Option 2: Lookup + analysis
                [
                    {"name": "lookup_sales_data", "args": {"query": "RELEVANT"}},
                    {"name": "analyze_sales_data", "args": {"data": "ANY"}}
                ]
            ]
        },
        {
            "question": "What was the total revenue across all stores?",
            "expected_tool_combinations": [
                [{"name": "lookup_sales_data", "args": {"query": "RELEVANT"}}],
                [
                    {"name": "lookup_sales_data", "args": {"query": "RELEVANT"}},
                    {"name": "analyze_sales_data", "args": {"data": "ANY"}}
                ]
            ]
        },
        {
            "question": "Which store had the highest sales volume?",
            "expected_tool_combinations": [
                [{"name": "lookup_sales_data", "args": {"query": "RELEVANT"}}],
                [
                    {"name": "lookup_sales_data", "args": {"query": "RELEVANT"}},
                    {"name": "analyze_sales_data", "args": {"data": "ANY"}}
                ]
            ]
        },
        {
            "question": "Create a bar chart showing total sales by store",
            "expected_tool_combinations": [
                # Option 1: Lookup + visualization
                [
                    {"name": "lookup_sales_data", "args": {"query": "RELEVANT"}},
                    {"name": "generate_visualization", "args": {"chart_type": "CHART_TYPE", "data": "ANY"}}
                ],
                # Option 2: Full workflow
                [
                    {"name": "lookup_sales_data", "args": {"query": "RELEVANT"}},
                    {"name": "generate_visualization", "args": {"chart_type": "CHART_TYPE", "data": "ANY"}},
                    {"name": "run_python_code", "args": {"code": "CODE"}}
                ]
            ]
        },
        {
            "question": "What percentage of items were sold on promotion?",
            "expected_tool_combinations": [
                [{"name": "lookup_sales_data", "args": {"query": "RELEVANT"}}],
                [
                    {"name": "lookup_sales_data", "args": {"query": "RELEVANT"}},
                    {"name": "analyze_sales_data", "args": {"data": "ANY"}}
                ]
            ]
        },
        {
            "question": "Plot daily sales volume over time",
            "expected_tool_combinations": [
                [
                    {"name": "lookup_sales_data", "args": {"query": "RELEVANT"}},
                    {"name": "generate_visualization", "args": {"chart_type": "CHART_TYPE", "data": "ANY"}}
                ],
                [
                    {"name": "lookup_sales_data", "args": {"query": "RELEVANT"}},
                    {"name": "generate_visualization", "args": {"chart_type": "CHART_TYPE", "data": "ANY"}},
                    {"name": "run_python_code", "args": {"code": "CODE"}}
                ]
            ]
        },
        {
            "question": "What was the average transaction value?",
            "expected_tool_combinations": [
                [{"name": "lookup_sales_data", "args": {"query": "RELEVANT"}}],
                [
                    {"name": "lookup_sales_data", "args": {"query": "RELEVANT"}},
                    {"name": "analyze_sales_data", "args": {"data": "ANY"}}
                ]
            ]
        },
        {
            "question": "Create a box plot of transaction values",
            "expected_tool_combinations": [
                [
                    {"name": "lookup_sales_data", "args": {"query": "RELEVANT"}},
                    {"name": "generate_visualization", "args": {"chart_type": "CHART_TYPE", "data": "ANY"}}
                ],
                [
                    {"name": "lookup_sales_data", "args": {"query": "RELEVANT"}},
                    {"name": "generate_visualization", "args": {"chart_type": "CHART_TYPE", "data": "ANY"}},
                    {"name": "run_python_code", "args": {"code": "CODE"}}
                ]
            ]
        },
        {
            "question": "Which products were frequently purchased together?",
            "expected_tool_combinations": [
                [{"name": "lookup_sales_data", "args": {"query": "RELEVANT"}}],
                [
                    {"name": "lookup_sales_data", "args": {"query": "RELEVANT"}},
                    {"name": "analyze_sales_data", "args": {"data": "ANY"}}
                ]
            ]
        },
        {
            "question": "Plot a line graph showing the sales trend over time with a 7-day moving average",
            "expected_tool_combinations": [
                [
                    {"name": "lookup_sales_data", "args": {"query": "RELEVANT"}},
                    {"name": "generate_visualization", "args": {"chart_type": "CHART_TYPE", "data": "ANY"}}
                ],
                [
                    {"name": "lookup_sales_data", "args": {"query": "RELEVANT"}},
                    {"name": "generate_visualization", "args": {"chart_type": "CHART_TYPE", "data": "ANY"}},
                    {"name": "run_python_code", "args": {"code": "CODE"}}
                ],
                [
                    {"name": "lookup_sales_data", "args": {"query": "RELEVANT"}},
                    {"name": "run_python_code", "args": {"code": "CODE"}}
                ]
            ]
        }
    ]
    
    # Convert to DataFrame format for Phoenix
    dataset_rows = []
    for item in realistic_ground_truth:
        # Create a simplified tool_calls string for backward compatibility
        primary_combination = item["expected_tool_combinations"][0]
        tool_names = [tool["name"] for tool in primary_combination]
        tool_calls_str = ", ".join(tool_names)
        
        dataset_rows.append({
            "question": item["question"],
            "tool_calls": tool_calls_str,  # Keep for backward compatibility
            "expected_tool_combinations": item["expected_tool_combinations"]  # Enhanced ground truth
        })
    
    realistic_df = pd.DataFrame(dataset_rows)
    
    # Upload to Phoenix
    dataset = px_client.upload_dataset(
        dataframe=realistic_df,
        dataset_name=f"realistic_tool_calling_ground_truth_{id}",
        input_keys=["question"],
        output_keys=["tool_calls", "expected_tool_combinations"],
    )
    
    return realistic_df, dataset

def main():
    """Main function using Phoenix's experiment framework with realistic ground truth."""
    
    print("=== Setting up Realistic Ground Truth Dataset ===")
    ground_truth_df, dataset = setup_realistic_ground_truth_dataset()
    print(f"Realistic dataset created with {len(ground_truth_df)} questions")
    
    print("\n=== Running Phoenix Experiment ===")
    
    # Run experiment with basic evaluators
    experiment = run_experiment(
        dataset,
        sales_agent_task,
        evaluators=[tool_accuracy_evaluator],
        experiment_name="Realistic Tool Calling Eval 2" \
        "",
        experiment_description="Evaluating tool calling with realistic expectations",
    )
    
    print(f"Initial experiment completed")
    
    print("\n=== Adding Enhanced Evaluators ===")
    
    # Add enhanced evaluators that use realistic ground truth
    experiment = evaluate_experiment(
        experiment, 
        evaluators=[
            tool_arguments_evaluator,
            tool_execution_order_evaluator,
            tool_argument_values_evaluator
        ]
    )
    
    print("\n=== Enhanced Evaluation Results ===")
    
    # Extract and display results
    if hasattr(experiment, 'get_results'):
        results_df = experiment.get_results()
    elif hasattr(experiment, 'dataframe'):
        results_df = experiment.dataframe
    else:
        print(f"Experiment type: {type(experiment)}")
        return experiment
    
    # Calculate aggregate metrics
    print(f"Total Questions: {len(results_df)}")
    print(f"Available columns: {list(results_df.columns)}")
    
    # Extract scores for all evaluators
    evaluator_scores = {}
    for col in results_df.columns:
        if any(eval_name in col.lower() for eval_name in ['accuracy', 'arguments', 'order', 'values']):
            scores = results_df[col].dropna()
            if len(scores) > 0:
                evaluator_scores[col] = {
                    'mean': scores.mean(),
                    'min': scores.min(),
                    'max': scores.max()
                }
    
    print("\n=== Aggregate Metrics ===")
    for eval_name, metrics in evaluator_scores.items():
        print(f"{eval_name}:")
        print(f"  Average: {metrics['mean']:.3f}")
        print(f"  Min: {metrics['min']:.3f}")
        print(f"  Max: {metrics['max']:.3f}")
    
    print("\n=== Detailed Sample Results ===")
    for i in range(min(3, len(results_df))):
        row = results_df.iloc[i]
        print(f"\n{i+1}. Question: {row.get('input.question', 'N/A')}")
        
        # Show scores for all evaluators
        for col in results_df.columns:
            if any(eval_name in col.lower() for eval_name in ['accuracy', 'arguments', 'order', 'values']):
                score = row.get(col, 'N/A')
                if score != 'N/A':
                    print(f"   {col}: {score:.3f}")
    
    return experiment

if __name__ == "__main__":
    results = main()



Overriding of current TracerProvider is not allowed
DependencyConflict: requested: "openai >= 1.69.0" but found: "None"
Attempting to instrument while already instrumented
Attempting to instrument while already instrumented
/Users/stepan.lavrinenko/phoenix/.venv/lib/python3.12/site-packages/phoenix/utilities/client.py:51: UserWarning: ⚠️⚠️ The Phoenix server (10.13.2) and client (11.5.0) versions are severely mismatched. Upgrade  either the client or server to ensure API compatibility ⚠️⚠️
  warnings.warn(
/Users/stepan.lavrinenko/phoenix/.venv/lib/python3.12/site-packages/phoenix/utilities/client.py:51: UserWarning: ⚠️⚠️ The Phoenix server (10.13.2) and client (11.5.0) versions are severely mismatched. Upgrade  either the client or server to ensure API compatibility ⚠️⚠️
  warnings.warn(
🐌!! If running inside a notebook, patching the event loop with nest_asyncio will allow asynchronous eval submission, and is significantly faster. To patch the event loop, run `nest_asyncio.apply()`.


🔭 OpenTelemetry Tracing Details 🔭
|  Phoenix Project: tool_calling_evaluation
|  Span Processor: SimpleSpanProcessor
|  Collector Endpoint: localhost:4317
|  Transport: gRPC
|  Transport Headers: {'user-agent': '****'}
|  
|  Using a default SpanProcessor. `add_span_processor` will overwrite this default.
|  
|  ⚠️ WARNING: It is strongly advised to use a BatchSpanProcessor in production environments.
|  
|  `register` has set this TracerProvider as the global OpenTelemetry default.
|  To disable this behavior, call `register` with `set_global_tracer_provider=False`.

=== Setting up Realistic Ground Truth Dataset ===
📤 Uploading dataset...
💾 Examples uploaded: http://localhost:6006/datasets/RGF0YXNldDoxNQ==/examples
🗄️ Dataset version ID: RGF0YXNldFZlcnNpb246MTQ=
Realistic dataset created with 10 questions

=== Running Phoenix Experiment ===
🧪 Experiment started.
📺 View dataset experiments: http://localhost:6006/datasets/RGF0YXNldDoxNQ==/experiments
🔗 View this experiment: http://local


/Users/stepan.lavrinenko/phoenix/.venv/lib/python3.12/site-packages/phoenix/utilities/client.py:51: UserWarning: ⚠️⚠️ The Phoenix server (10.13.2) and client (11.5.0) versions are severely mismatched. Upgrade  either the client or server to ensure API compatibility ⚠️⚠️
  warnings.warn(



Question: What was the most popular product SKU?
Extracted tool calls: ['lookup_sales_data']
Tool call details: [{'name': 'lookup_sales_data', 'args': {'query': 'SELECT sku, COUNT(*) AS sales_count FROM sales GROUP BY sku ORDER BY sales_count DESC LIMIT 1'}, 'call_id': '4c040b93-1b68-418b-94d7-53fab924f1a4', 'order': 1}]
---


Question: What was the total revenue across all stores?
Extracted tool calls: ['lookup_sales_data']
Tool call details: [{'name': 'lookup_sales_data', 'args': {'query': 'total revenue across all stores'}, 'call_id': 'd50f5679-772a-4757-9a23-d78a31dba7d5', 'order': 1}]
---


...

Question: Plot a line graph showing the sales trend over time with a 7-day moving average
Extracted tool calls: ['lookup_sales_data', 'run_python_code', 'generate_visualization']
Tool call details: [{'name': 'lookup_sales_data', 'args': {'query': 'sales_data_over_time'}, 'call_id': 'f2a9936b-7348-41fd-b8c2-60045caece1e', 'order': 1}, {'name': 'run_python_code', 'args': {'code': '\nimport pandas as pd\nsales_data = pd.read_json(\'{"date":["2024-01-01","2024-01-02","2024-01-03","2024-01-04","2024-01-05","2024-01-06","2024-01-07","2024-01-08"],"sales":[10,12,15,14,18,20,22,25]}\')\nsales_data["date"] = pd.to_datetime(sales_data["date"])\nsales_data["7_day_avg"] = sales_data["sales"].rolling(window=7).mean()\nresult = sales_data.to_json(orient="records")\nprint(result)\n\n'}, 'call_id': 'ee823279-a1e7-4a21-86dd-85e68ba6c4a1', 'order': 2}, {'name': 'generate_visualization', 'args': {'chart_type': 'line', 'data': '[{"date":"2024-01-01T00:00:00.000Z","sales":10,"7_day_avg":null},{"date":"2024


/Users/stepan.lavrinenko/phoenix/.venv/lib/python3.12/site-packages/phoenix/utilities/client.py:51: UserWarning: ⚠️⚠️ The Phoenix server (10.13.2) and client (11.5.0) versions are severely mismatched. Upgrade  either the client or server to ensure API compatibility ⚠️⚠️
  warnings.warn(
running experiment evaluations |██████████| 10/10 (100.0%) | ⏳ 00:00<00:00 | 185.44it/s
🐌!! If running inside a notebook, patching the event loop with nest_asyncio will allow asynchronous eval submission, and is significantly faster. To patch the event loop, run `nest_asyncio.apply()`.



🔗 View this experiment: http://localhost:6006/datasets/RGF0YXNldDoxNQ==/compare?experimentId=RXhwZXJpbWVudDoxMA==

Experiment Summary (07/10/25 10:57 AM -0600)
--------------------------------------------
                 evaluator   n  n_scores  avg_score
0  tool_accuracy_evaluator  10        10        0.9

Tasks Summary (07/10/25 10:57 AM -0600)
---------------------------------------
   n_examples  n_runs  n_errors
0          10      10         0
Initial experiment completed

=== Adding Enhanced Evaluators ===
🧠 Evaluation started.



/Users/stepan.lavrinenko/phoenix/.venv/lib/python3.12/site-packages/phoenix/utilities/client.py:51: UserWarning: ⚠️⚠️ The Phoenix server (10.13.2) and client (11.5.0) versions are severely mismatched. Upgrade  either the client or server to ensure API compatibility ⚠️⚠️
  warnings.warn(

running experiment evaluations |██████████| 30/30 (100.0%) | ⏳ 00:00<00:00 | 150.18it/s



🔗 View this experiment: http://localhost:6006/datasets/RGF0YXNldDoxNQ==/compare?experimentId=RXhwZXJpbWVudDoxMA==

Experiment Summary (07/10/25 10:57 AM -0600)
--------------------------------------------
                        evaluator   n  n_scores  avg_score
0  tool_argument_values_evaluator  10        10   0.613333
1        tool_arguments_evaluator  10        10   0.900000
2  tool_execution_order_evaluator  10        10   0.890000

Experiment Summary (07/10/25 10:57 AM -0600)
--------------------------------------------
                 evaluator   n  n_scores  avg_score
0  tool_accuracy_evaluator  10        10        0.9

Tasks Summary (07/10/25 10:57 AM -0600)
---------------------------------------
   n_examples  n_runs  n_errors
0          10      10         0

=== Enhanced Evaluation Results ===
Experiment type: <class 'phoenix.experiments.types.RanExperiment'>


## Add explanations to evaluation scores

In [ ]:
import os
import uuid
import pandas as pd
from typing import List, Dict, Any, Optional, Set, Tuple
from langchain_google_vertexai import ChatVertexAI
from langchain_core.messages import HumanMessage, AIMessage, SystemMessage
from langchain_core.messages.utils import filter_messages
from langchain_core.tools import tool
from langgraph.prebuilt import ToolNode
from langgraph.graph import StateGraph, END, START
from langgraph.graph.message import add_messages
from typing_extensions import Annotated, TypedDict
from phoenix.otel import register
from openinference.instrumentation.langchain import LangChainInstrumentor
from openinference.instrumentation import using_attributes, using_tags
from phoenix.experiments import evaluate_experiment, run_experiment
from phoenix.evals import GeminiModel, llm_classify
import json
import phoenix as px

# Phoenix configuration
project_name = "tool_calling_evaluation"

os.environ["PHOENIX_COLLECTOR_ENDPOINT"] = "http://localhost:6006"
os.environ["PHOENIX_PROJECT_NAME"] = project_name

# Configure the Phoenix tracer
tracer_provider = register(
    project_name=project_name,
    auto_instrument=True
)

# Import the automatic instrumentor from OpenInference
LangChainInstrumentor().instrument(tracer_provider=tracer_provider)
from openinference.instrumentation import using_attributes, using_tags

tracer = tracer_provider.get_tracer(__name__)

# Initialize ChatVertexAI model
client = ChatVertexAI(
    model_name="gemini-1.5-pro",
    temperature=0.1
)

# Initialize Phoenix eval model
eval_model = GeminiModel(
    model="gemini-1.5-pro"
)

# Initialize Phoenix client
px_client = px.Client()

# Global storage for explanations (to be accessed later)
evaluation_explanations = {}

# Define evaluation prompt templates
TOOL_ACCURACY_TEMPLATE = """
You are evaluating whether an AI agent selected the correct tools for a given question.

Question: {question}
Expected Tools: {expected_tools}
Actual Tools Called: {actual_tools}

Evaluate if the agent selected appropriate tools that can answer the question effectively.
Consider that multiple tool combinations may be acceptable for complex tasks.

Classify as "correct" or "incorrect".
"""

TOOL_ARGUMENTS_TEMPLATE = """
You are evaluating whether an AI agent provided appropriate arguments when calling tools.

Question: {question}
Tool Calls with Arguments: {tool_calls_with_args}

Evaluate if the arguments are relevant, complete, and appropriate for the tools called.

Classify as "correct" or "incorrect".
"""

TOOL_ORDER_TEMPLATE = """
You are evaluating whether tools were executed in a logical order.

Question: {question}
Tool Execution Order: {execution_order}

Evaluate if the tools were executed in a logical sequence that respects dependencies.
Data lookup should typically come before analysis or visualization.

Classify as "correct" or "incorrect".
"""

TOOL_VALUES_TEMPLATE = """
You are evaluating the quality of argument values provided to tools.

Question: {question}
Tool Arguments and Values: {tool_args_values}

Evaluate if the argument values are relevant, appropriate, and high-quality for the question.

Classify as "correct" or "incorrect".
"""

# Define actual tools for the sales agent
@tool
def lookup_sales_data(query: str) -> str:
    """Look up sales data from the database based on a query."""
    return f"Sales data retrieved for query: {query}"

@tool
def analyze_sales_data(data: str) -> str:
    """Analyze sales data and provide insights."""
    return f"Analysis completed on: {data}"

@tool
def generate_visualization(chart_type: str, data: str) -> str:
    """Generate a visualization based on chart type and data."""
    return f"Generated {chart_type} visualization for: {data}"

@tool
def run_python_code(code: str) -> str:
    """Execute Python code for data processing."""
    return f"Executed Python code: {code}"

# Simplified agent state - only messages
class AgentState(TypedDict):
    messages: Annotated[list, add_messages]

# Helper function to extract tool information from messages
def extract_tool_calls_from_messages(messages: List[Any]) -> Dict[str, Any]:
    """Extract tool calls and arguments from message history using filter_messages."""
    tool_calls = []
    tool_call_details = []
    execution_order = []
    
    # Filter for AI messages that contain tool calls
    ai_messages = filter_messages(messages, include_types=[AIMessage])
    
    for message in ai_messages:
        if hasattr(message, 'tool_calls') and message.tool_calls:
            for tool_call in message.tool_calls:
                tool_name = tool_call["name"]
                tool_args = tool_call.get("args", {})
                
                tool_calls.append(tool_name)
                execution_order.append(tool_name)
                
                tool_call_details.append({
                    "name": tool_name,
                    "args": tool_args,
                    "call_id": tool_call.get("id", ""),
                    "order": len(execution_order)
                })
        
        elif hasattr(message, 'additional_kwargs') and message.additional_kwargs:
            additional_kwargs = message.additional_kwargs
            if 'tool_calls' in additional_kwargs:
                for tool_call in additional_kwargs['tool_calls']:
                    if isinstance(tool_call, dict):
                        if 'function' in tool_call:
                            tool_name = tool_call['function']['name']
                            try:
                                tool_args = json.loads(tool_call['function']['arguments'])
                            except:
                                tool_args = {}
                        else:
                            tool_name = tool_call.get('name', '')
                            tool_args = tool_call.get('args', {})
                        
                        if tool_name:
                            tool_calls.append(tool_name)
                            execution_order.append(tool_name)
                            
                            tool_call_details.append({
                                "name": tool_name,
                                "args": tool_args,
                                "call_id": tool_call.get("id", ""),
                                "order": len(execution_order)
                            })
    
    # Also check for tool messages
    from langchain_core.messages import ToolMessage
    tool_messages = filter_messages(messages, include_types=[ToolMessage])
    
    for tool_msg in tool_messages:
        if hasattr(tool_msg, 'name') and tool_msg.name:
            tool_name = tool_msg.name
            if tool_name not in tool_calls:
                tool_calls.append(tool_name)
                execution_order.append(tool_name)
                
                tool_call_details.append({
                    "name": tool_name,
                    "args": {},
                    "call_id": getattr(tool_msg, 'tool_call_id', ''),
                    "order": len(execution_order)
                })
    
    return {
        "tool_calls": tool_calls,
        "tool_call_details": tool_call_details,
        "execution_order": execution_order
    }

# Create the sales agent
def create_sales_agent():
    """Create a tool-calling sales agent using LangGraph."""
    
    tools = [lookup_sales_data, analyze_sales_data, generate_visualization, run_python_code]
    tool_node = ToolNode(tools)
    model_with_tools = client.bind_tools(tools)
    
    def should_continue(state: AgentState) -> str:
        messages = state["messages"]
        last_message = messages[-1]
        
        if hasattr(last_message, 'tool_calls') and last_message.tool_calls:
            return "tools"
        return END
    
    def call_model(state: AgentState) -> AgentState:
        messages = state["messages"]
        response = model_with_tools.invoke(messages)
        return {"messages": [response]}
    
    def call_tools(state: AgentState) -> AgentState:
        messages = state["messages"]
        last_message = messages[-1]
        tool_outputs = tool_node.invoke({"messages": [last_message]})
        return {"messages": tool_outputs["messages"]}
    
    workflow = StateGraph(AgentState)
    workflow.add_node("agent", call_model)
    workflow.add_node("tools", call_tools)
    workflow.add_edge(START, "agent")
    workflow.add_conditional_edges("agent", should_continue, {"tools": "tools", END: END})
    workflow.add_edge("tools", "agent")
    
    return workflow.compile()

# Task function
def sales_agent_task(input_data: Dict[str, Any]) -> Dict[str, Any]:
    """Task function that runs the sales agent on a question."""
    question = input_data["question"]
    agent_app = create_sales_agent()
    
    with using_attributes(user_id="experiment_user", session_id="phoenix_experiment"):
        with using_tags(["phoenix_experiment", "tool_calling"]):
            initial_state = {"messages": [HumanMessage(content=question)]}
            final_state = agent_app.invoke(initial_state)
            tool_info = extract_tool_calls_from_messages(final_state["messages"])
            
            final_message = final_state["messages"][-1]
            response_content = final_message.content if hasattr(final_message, 'content') else str(final_message)
            
            return {
                "response": response_content,
                "messages": final_state["messages"],
                "tool_calls": ", ".join(tool_info["tool_calls"]) if tool_info["tool_calls"] else "",
                "individual_tool_calls": tool_info["tool_calls"],
                "tool_call_details": tool_info["tool_call_details"],
                "execution_order": tool_info["execution_order"]
            }

# Helper function to store explanations with unique keys
def store_explanation(question: str, evaluator_name: str, explanation: str, score: float):
    """Store explanation for later retrieval."""
    key = f"{question}_{evaluator_name}"
    evaluation_explanations[key] = {
        "explanation": explanation,
        "score": score,
        "evaluator": evaluator_name
    }

# LLM-based evaluators that capture explanations
def tool_accuracy_evaluator(output: Dict[str, Any], expected: Dict[str, Any]) -> float:
    """Evaluate tool calling accuracy using LLM classification."""
    # Extract tool calls from messages
    messages = output.get("messages", [])
    tool_info = extract_tool_calls_from_messages(messages)
    actual_tools = tool_info["tool_calls"]
    
    # Get expected tools from ground truth
    expected_combinations = expected.get("expected_tool_combinations", [])
    expected_tools_list = []
    for combination in expected_combinations:
        tools = [tool["name"] for tool in combination]
        expected_tools_list.append(", ".join(tools))
    
    question = expected.get("question", "")
    
    # Create evaluation dataframe
    eval_data = pd.DataFrame([{
        "question": question,
        "expected_tools": " OR ".join(expected_tools_list),
        "actual_tools": ", ".join(actual_tools) if actual_tools else "None"
    }])
    
    # Use LLM classification
    try:
        response_classifications = llm_classify(
            dataframe=eval_data,
            template=TOOL_ACCURACY_TEMPLATE,
            model=eval_model,
            rails=["correct", "incorrect"],
            provide_explanation=True,
        )
        
        if len(response_classifications) > 0:
            result = response_classifications.iloc[0]
            score = 1.0 if result.get("label") == "correct" else 0.0
            explanation = result.get("explanation", "No explanation provided")
            
            # Store explanation for later display
            store_explanation(question, "tool_accuracy", explanation, score)
            
            print(f"[TOOL ACCURACY] Question: {question}")
            print(f"[TOOL ACCURACY] Score: {score:.2f}")
            print(f"[TOOL ACCURACY] Explanation: {explanation}")
            print("---")
            
            return score
    except Exception as e:
        print(f"Error in tool_accuracy_evaluator: {e}")
        store_explanation(question, "tool_accuracy", f"Error: {e}", 0.0)
        return 0.0
    
    return 0.0

def tool_arguments_evaluator(output: Dict[str, Any], expected: Dict[str, Any]) -> float:
    """Evaluate tool arguments using LLM classification."""
    # Extract tool calls from messages
    messages = output.get("messages", [])
    tool_info = extract_tool_calls_from_messages(messages)
    tool_call_details = tool_info["tool_call_details"]
    
    question = expected.get("question", "")
    
    # Format tool calls with arguments for evaluation
    tool_calls_with_args = []
    for tool_detail in tool_call_details:
        tool_name = tool_detail.get("name", "")
        tool_args = tool_detail.get("args", {})
        tool_calls_with_args.append(f"{tool_name}({tool_args})")
    
    # Create evaluation dataframe
    eval_data = pd.DataFrame([{
        "question": question,
        "tool_calls_with_args": "; ".join(tool_calls_with_args) if tool_calls_with_args else "No tool calls"
    }])
    
    # Use LLM classification
    try:
        response_classifications = llm_classify(
            dataframe=eval_data,
            template=TOOL_ARGUMENTS_TEMPLATE,
            model=eval_model,
            rails=["correct", "incorrect"],
            provide_explanation=True,
        )
        
        if len(response_classifications) > 0:
            result = response_classifications.iloc[0]
            score = 1.0 if result.get("label") == "correct" else 0.0
            explanation = result.get("explanation", "No explanation provided")
            
            # Store explanation for later display
            store_explanation(question, "tool_arguments", explanation, score)
            
            print(f"[TOOL ARGUMENTS] Question: {question}")
            print(f"[TOOL ARGUMENTS] Score: {score:.2f}")
            print(f"[TOOL ARGUMENTS] Explanation: {explanation}")
            print("---")
            
            return score
    except Exception as e:
        print(f"Error in tool_arguments_evaluator: {e}")
        store_explanation(question, "tool_arguments", f"Error: {e}", 0.0)
        return 0.0
    
    return 0.0

def tool_execution_order_evaluator(output: Dict[str, Any], expected: Dict[str, Any]) -> float:
    """Evaluate tool execution order using LLM classification."""
    # Extract execution order from messages
    messages = output.get("messages", [])
    tool_info = extract_tool_calls_from_messages(messages)
    execution_order = tool_info["execution_order"]
    
    question = expected.get("question", "")
    
    # Create evaluation dataframe
    eval_data = pd.DataFrame([{
        "question": question,
        "execution_order": " → ".join(execution_order) if execution_order else "No tools executed"
    }])
    
    # Use LLM classification
    try:
        response_classifications = llm_classify(
            dataframe=eval_data,
            template=TOOL_ORDER_TEMPLATE,
            model=eval_model,
            rails=["correct", "incorrect"],
            provide_explanation=True,
        )
        
        if len(response_classifications) > 0:
            result = response_classifications.iloc[0]
            score = 1.0 if result.get("label") == "correct" else 0.0
            explanation = result.get("explanation", "No explanation provided")
            
            # Store explanation for later display
            store_explanation(question, "tool_execution_order", explanation, score)
            
            print(f"[TOOL EXECUTION ORDER] Question: {question}")
            print(f"[TOOL EXECUTION ORDER] Score: {score:.2f}")
            print(f"[TOOL EXECUTION ORDER] Explanation: {explanation}")
            print("---")
            
            return score
    except Exception as e:
        print(f"Error in tool_execution_order_evaluator: {e}")
        store_explanation(question, "tool_execution_order", f"Error: {e}", 0.0)
        return 0.0
    
    return 0.0

def tool_argument_values_evaluator(output: Dict[str, Any], expected: Dict[str, Any]) -> float:
    """Evaluate tool argument values using LLM classification."""
    # Extract tool calls from messages
    messages = output.get("messages", [])
    tool_info = extract_tool_calls_from_messages(messages)
    tool_call_details = tool_info["tool_call_details"]
    
    question = expected.get("question", "")
    
    # Format tool arguments and values for evaluation
    tool_args_values = []
    for tool_detail in tool_call_details:
        tool_name = tool_detail.get("name", "")
        tool_args = tool_detail.get("args", {})
        for arg_name, arg_value in tool_args.items():
            tool_args_values.append(f"{tool_name}.{arg_name} = '{arg_value}'")
    
    # Create evaluation dataframe
    eval_data = pd.DataFrame([{
        "question": question,
        "tool_args_values": "; ".join(tool_args_values) if tool_args_values else "No arguments provided"
    }])
    
    # Use LLM classification
    try:
        response_classifications = llm_classify(
            dataframe=eval_data,
            template=TOOL_VALUES_TEMPLATE,
            model=eval_model,
            rails=["correct", "incorrect"],
            provide_explanation=True,
        )
        
        if len(response_classifications) > 0:
            result = response_classifications.iloc[0]
            score = 1.0 if result.get("label") == "correct" else 0.0
            explanation = result.get("explanation", "No explanation provided")
            
            # Store explanation for later display
            store_explanation(question, "tool_argument_values", explanation, score)
            
            print(f"[TOOL ARGUMENT VALUES] Question: {question}")
            print(f"[TOOL ARGUMENT VALUES] Score: {score:.2f}")
            print(f"[TOOL ARGUMENT VALUES] Explanation: {explanation}")
            print("---")
            
            return score
    except Exception as e:
        print(f"Error in tool_argument_values_evaluator: {e}")
        store_explanation(question, "tool_argument_values", f"Error: {e}", 0.0)
        return 0.0
    
    return 0.0

def setup_realistic_ground_truth_dataset():
    """Setup realistic ground truth dataset."""
    
    id = str(uuid.uuid4())
    
    realistic_ground_truth = [
        {
            "question": "What was the most popular product SKU?",
            "expected_tool_combinations": [
                [{"name": "lookup_sales_data", "args": {"query": "RELEVANT"}}],
                [
                    {"name": "lookup_sales_data", "args": {"query": "RELEVANT"}},
                    {"name": "analyze_sales_data", "args": {"data": "ANY"}}
                ]
            ]
        },
        {
            "question": "Create a bar chart showing total sales by store",
            "expected_tool_combinations": [
                [
                    {"name": "lookup_sales_data", "args": {"query": "RELEVANT"}},
                    {"name": "generate_visualization", "args": {"chart_type": "CHART_TYPE", "data": "ANY"}}
                ],
                [
                    {"name": "lookup_sales_data", "args": {"query": "RELEVANT"}},
                    {"name": "generate_visualization", "args": {"chart_type": "CHART_TYPE", "data": "ANY"}},
                    {"name": "run_python_code", "args": {"code": "CODE"}}
                ]
            ]
        },
        {
            "question": "What was the total revenue across all stores?",
            "expected_tool_combinations": [
                [{"name": "lookup_sales_data", "args": {"query": "RELEVANT"}}],
                [
                    {"name": "lookup_sales_data", "args": {"query": "RELEVANT"}},
                    {"name": "analyze_sales_data", "args": {"data": "ANY"}}
                ]
            ]
        }
    ]
    
    dataset_rows = []
    for item in realistic_ground_truth:
        primary_combination = item["expected_tool_combinations"][0]
        tool_names = [tool["name"] for tool in primary_combination]
        tool_calls_str = ", ".join(tool_names)
        
        dataset_rows.append({
            "question": item["question"],
            "tool_calls": tool_calls_str,
            "expected_tool_combinations": item["expected_tool_combinations"]
        })
    
    realistic_df = pd.DataFrame(dataset_rows)
    
    dataset = px_client.upload_dataset(
        dataframe=realistic_df,
        dataset_name=f"explained_tool_calling_ground_truth_{id}",
        input_keys=["question"],
        output_keys=["tool_calls", "expected_tool_combinations"],
    )
    
    return realistic_df, dataset

def print_comprehensive_evaluation_summary():
    """Print a comprehensive summary of all evaluations with explanations."""
    
    print("\n" + "="*80)
    print("COMPREHENSIVE EVALUATION SUMMARY WITH EXPLANATIONS")
    print("="*80)
    
    if not evaluation_explanations:
        print("No evaluation explanations available.")
        return
    
    # Group explanations by question
    questions = set()
    for key in evaluation_explanations.keys():
        question = key.rsplit("_", 1)[0]  # Remove evaluator name from key
        questions.add(question)
    
    for i, question in enumerate(sorted(questions), 1):
        print(f"\n{'='*60}")
        print(f"QUESTION {i}: {question}")
        print(f"{'='*60}")
        
        # Find all evaluations for this question
        evaluators = ["tool_accuracy", "tool_arguments", "tool_execution_order", "tool_argument_values"]
        
        for evaluator in evaluators:
            key = f"{question}_{evaluator}"
            if key in evaluation_explanations:
                eval_data = evaluation_explanations[key]
                score = eval_data["score"]
                explanation = eval_data["explanation"]
                
                print(f"\n📊 {evaluator.upper().replace('_', ' ')}:")
                print(f"   Score: {score:.2f}/1.0 ({'✅ CORRECT' if score >= 0.5 else '❌ INCORRECT'})")
                print(f"   Explanation: {explanation}")
        
        print(f"\n{'-'*60}")
    
    # Calculate and display aggregate metrics
    print(f"\n{'='*60}")
    print("AGGREGATE METRICS")
    print(f"{'='*60}")
    
    evaluator_scores = {
        "tool_accuracy": [],
        "tool_arguments": [],
        "tool_execution_order": [],
        "tool_argument_values": []
    }
    
    for key, data in evaluation_explanations.items():
        evaluator = data["evaluator"]
        score = data["score"]
        if evaluator in evaluator_scores:
            evaluator_scores[evaluator].append(score)
    
    for evaluator, scores in evaluator_scores.items():
        if scores:
            avg_score = sum(scores) / len(scores)
            print(f"\n📈 {evaluator.upper().replace('_', ' ')}:")
            print(f"   Average Score: {avg_score:.3f}")
            print(f"   Success Rate: {sum(1 for s in scores if s >= 0.5) / len(scores):.1%}")
            print(f"   Total Evaluations: {len(scores)}")

def main():
    """Main function with proper LLM-based evaluation and explanation display."""
    
    print("=== Setting up Ground Truth Dataset ===")
    ground_truth_df, dataset = setup_realistic_ground_truth_dataset()
    print(f"Dataset created with {len(ground_truth_df)} questions")
    
    print("\n=== Running Phoenix Experiment ===")
    
    # Clear previous explanations
    evaluation_explanations.clear()
    
    # Run experiment with real LLM-based evaluators
    experiment = run_experiment(
        dataset,
        sales_agent_task,
        evaluators=[tool_accuracy_evaluator],
        experiment_name="Explained Tool Calling Eval",
        experiment_description="Tool calling evaluation with visible explanations",
    )
    
    print("\n=== Adding Additional LLM Evaluators ===")
    
    # Add additional LLM-based evaluators
    experiment = evaluate_experiment(
        experiment, 
        evaluators=[
            tool_arguments_evaluator,
            tool_execution_order_evaluator,
            tool_argument_values_evaluator
        ]
    )
    
    print("\n=== Phoenix Experiment Results ===")
    
    # Extract and display standard Phoenix results
    if hasattr(experiment, 'get_results'):
        results_df = experiment.get_results()
    elif hasattr(experiment, 'dataframe'):
        results_df = experiment.dataframe
    else:
        print(f"Experiment type: {type(experiment)}")
        results_df = None
    
    if results_df is not None:
        print(f"Total Questions: {len(results_df)}")
        
        # Extract scores for all evaluators
        evaluator_scores = {}
        for col in results_df.columns:
            if any(eval_name in col.lower() for eval_name in ['accuracy', 'arguments', 'order', 'values']):
                scores = results_df[col].dropna()
                if len(scores) > 0:
                    evaluator_scores[col] = {
                        'mean': scores.mean(),
                        'min': scores.min(),
                        'max': scores.max()
                    }
        
        print("\n=== Phoenix Aggregate Metrics ===")
        for eval_name, metrics in evaluator_scores.items():
            print(f"{eval_name}:")
            print(f"  Average: {metrics['mean']:.3f}")
            print(f"  Min: {metrics['min']:.3f}")
            print(f"  Max: {metrics['max']:.3f}")
    
    # Print comprehensive evaluation summary with explanations
    print_comprehensive_evaluation_summary()
    
    return experiment

if __name__ == "__main__":
    experiment = main()


Overriding of current TracerProvider is not allowed
Attempting to instrument while already instrumented
Attempting to instrument while already instrumented
Attempting to instrument while already instrumented
/Users/stepan.lavrinenko/phoenix/.venv/lib/python3.12/site-packages/vertexai/generative_models/_generative_models.py:433: UserWarning: This feature is deprecated as of June 24, 2025 and will be removed on June 24, 2026. For details, see https://cloud.google.com/vertex-ai/generative-ai/docs/deprecations/genai-vertexai-sdk.
  warning_logs.show_deprecation_warning()
/Users/stepan.lavrinenko/phoenix/.venv/lib/python3.12/site-packages/phoenix/utilities/client.py:51: UserWarning: ⚠️⚠️ The Phoenix server (10.13.2) and client (11.5.0) versions are severely mismatched. Upgrade  either the client or server to ensure API compatibility ⚠️⚠️
  warnings.warn(
/Users/stepan.lavrinenko/phoenix/.venv/lib/python3.12/site-packages/phoenix/utilities/client.py:51: UserWarning: ⚠️⚠️ The Phoenix server (

🔭 OpenTelemetry Tracing Details 🔭
|  Phoenix Project: tool_calling_evaluation
|  Span Processor: SimpleSpanProcessor
|  Collector Endpoint: localhost:4317
|  Transport: gRPC
|  Transport Headers: {'user-agent': '****'}
|  
|  Using a default SpanProcessor. `add_span_processor` will overwrite this default.
|  
|  ⚠️ WARNING: It is strongly advised to use a BatchSpanProcessor in production environments.
|  
|  `register` has set this TracerProvider as the global OpenTelemetry default.
|  To disable this behavior, call `register` with `set_global_tracer_provider=False`.

=== Setting up Ground Truth Dataset ===
📤 Uploading dataset...
💾 Examples uploaded: http://localhost:6006/datasets/RGF0YXNldDoxNw==/examples
🗄️ Dataset version ID: RGF0YXNldFZlcnNpb246MTY=
Dataset created with 3 questions

=== Running Phoenix Experiment ===
🧪 Experiment started.
📺 View dataset experiments: http://localhost:6006/datasets/RGF0YXNldDoxNw==/experiments
🔗 View this experiment: http://localhost:6006/datasets/RG

running tasks |          | 0/3 (0.0%) | ⏳ 00:00<? | ?it/s/Users/stepan.lavrinenko/phoenix/.venv/lib/python3.12/site-packages/phoenix/utilities/client.py:51: UserWarning: ⚠️⚠️ The Phoenix server (10.13.2) and client (11.5.0) versions are severely mismatched. Upgrade  either the client or server to ensure API compatibility ⚠️⚠️
  warnings.warn(
running tasks |███▎      | 1/3 (33.3%) | ⏳ 00:05<00:10 |  5.17s/it/Users/stepan.lavrinenko/phoenix/.venv/lib/python3.12/site-packages/phoenix/utilities/client.py:51: UserWarning: ⚠️⚠️ The Phoenix server (10.13.2) and client (11.5.0) versions are severely mismatched. Upgrade  either the client or server to ensure API compatibility ⚠️⚠️
  warnings.warn(
running tasks |██████▋   | 2/3 (66.7%) | ⏳ 00:09<00:04 |  4.48s/it/Users/stepan.lavrinenko/phoenix/.venv/lib/python3.12/site-packages/phoenix/utilities/client.py:51: UserWarning: ⚠️⚠️ The Phoenix server (10.13.2) and client (11.5.0) versions are severely mismatched. Upgrade  either the client or serv

✅ Task runs completed.
🧠 Evaluation started.


running experiment evaluations |          | 0/3 (0.0%) | ⏳ 00:00<? | ?it/s/var/folders/z8/r7gby1gs0ts1089jpzfjwfjc0000gp/T/ipykernel_60987/1529874745.py:288: DeprecationWarning: `dataframe` argument is deprecated; use `data` instead
  response_classifications = llm_classify(
/Users/stepan.lavrinenko/phoenix/.venv/lib/python3.12/site-packages/vertexai/generative_models/_generative_models.py:433: UserWarning: This feature is deprecated as of June 24, 2025 and will be removed on June 24, 2026. For details, see https://cloud.google.com/vertex-ai/generative-ai/docs/deprecations/genai-vertexai-sdk.
  warning_logs.show_deprecation_warning()
🐌!! If running inside a notebook, patching the event loop with nest_asyncio will allow asynchronous eval submission, and is significantly faster. To patch the event loop, run `nest_asyncio.apply()`.
llm_classify |██████████| 1/1 (100.0%) | ⏳ 00:02<00:00 |  2.36s/it
/Users/stepan.lavrinenko/phoenix/.venv/lib/python3.12/site-packages/phoenix/utilities/client


🔗 View this experiment: http://localhost:6006/datasets/RGF0YXNldDoxNw==/compare?experimentId=RXhwZXJpbWVudDoxMg==

Experiment Summary (07/10/25 03:13 PM -0600)
--------------------------------------------
                 evaluator  n  n_scores  avg_score
0  tool_accuracy_evaluator  3         3        1.0

Tasks Summary (07/10/25 03:13 PM -0600)
---------------------------------------
   n_examples  n_runs  n_errors
0           3       3         0

=== Adding Additional LLM Evaluators ===
🧠 Evaluation started.


running experiment evaluations |          | 0/9 (0.0%) | ⏳ 00:00<? | ?it/s/var/folders/z8/r7gby1gs0ts1089jpzfjwfjc0000gp/T/ipykernel_60987/1529874745.py:330: DeprecationWarning: `dataframe` argument is deprecated; use `data` instead
  response_classifications = llm_classify(
/Users/stepan.lavrinenko/phoenix/.venv/lib/python3.12/site-packages/vertexai/generative_models/_generative_models.py:433: UserWarning: This feature is deprecated as of June 24, 2025 and will be removed on June 24, 2026. For details, see https://cloud.google.com/vertex-ai/generative-ai/docs/deprecations/genai-vertexai-sdk.
  warning_logs.show_deprecation_warning()
🐌!! If running inside a notebook, patching the event loop with nest_asyncio will allow asynchronous eval submission, and is significantly faster. To patch the event loop, run `nest_asyncio.apply()`.
llm_classify |██████████| 1/1 (100.0%) | ⏳ 00:01<00:00 |  1.44s/it
/Users/stepan.lavrinenko/phoenix/.venv/lib/python3.12/site-packages/phoenix/utilities/client


🔗 View this experiment: http://localhost:6006/datasets/RGF0YXNldDoxNw==/compare?experimentId=RXhwZXJpbWVudDoxMg==

Experiment Summary (07/10/25 03:13 PM -0600)
--------------------------------------------
                        evaluator  n  n_scores  avg_score
0  tool_argument_values_evaluator  3         3   0.666667
1        tool_arguments_evaluator  3         3   0.666667
2  tool_execution_order_evaluator  3         3   1.000000

Experiment Summary (07/10/25 03:13 PM -0600)
--------------------------------------------
                 evaluator  n  n_scores  avg_score
0  tool_accuracy_evaluator  3         3        1.0

Tasks Summary (07/10/25 03:13 PM -0600)
---------------------------------------
   n_examples  n_runs  n_errors
0           3       3         0

=== Final Results ===
Experiment type: <class 'phoenix.experiments.types.RanExperiment'>
